# 🛍️ ShopTalk — Phase 9: Testing & Evaluation

Phase 9 measures **retrieval**, **generation** and **latency** separately, then writes up what failed.

| § | What | Plan item |
|---|---|---|
| 1 | Setup: Drive, config, pinned deps, local runtime copy (Chroma on local disk) | — |
| 2 | Evaluation package written to `src/eval/` with `%%writefile` | — |
| 3 | Evaluation set: the **Phase 6 query set + labels**, plus thumbs-down / no-match cases mined from `logs/` | logs → eval cases |
| 4 | Retrieval precision, **pretrained vs LoRA fine-tuned E5** (same metrics as Phase 6) | 9.1 |
| 5 | Start the Phase 7 API as a subprocess, wait for `/health` = ok | — |
| 6 | End-to-end eval through `/recommend`: retrieval vs Phase 6, answer auto-checks, re-run of log cases | 9.1 |
| 7 | Manual rating of 30–50 answers (relevance, groundedness, fluency) | 9.2 |
| 8 | Latency: 120 requests, P50/P95/P99 per stage + streaming first-token/full-answer | 9.3 |
| 9 | `max_new_tokens` sweep: can full-answer P95 drop under 5 s without hurting quality? | Phase 8 follow-up |
| 10 | Failure cases + improvement plan (`failure_cases.md`) | 9.4 |
| 11 | Exit checklist → `results/phase9/` | — |

**How to run:** GPU runtime (T4 or better) → run top to bottom. Sections 7.2 and 9.3 are interactive (rating widgets);
everything else is unattended. The rating sheets live on Drive, so you can stop, come back and resume.
Expect ~45–60 min end to end without the manual rating.

**Auto-detection.** Paths the notebook cannot know in advance (Phase 6 eval file, Chroma dir, adapter dir,
FastAPI app module, Phase 8 example queries, the `max_new_tokens` key) are auto-detected and printed.
If a detection is wrong, set the matching override in the config cell (1.2); every override defaults to `None`.

**Logs hygiene.** All Phase 9 traffic uses `session_id` = `phase9-…`, and the API runs from a local copy, so
Phase 9 requests go to `/content/shoptalk_run/logs/` and never into the production `logs/` on Drive.

## 1. Setup

### 📘 1.1 Check the GPU and mount Google Drive

**Purpose:** Confirm a CUDA GPU runtime (a TPU runtime cannot run this pipeline), then mount Drive (Colab lesson learned) so every later cell can read the project folder. On a mount failure the cell prints the usual fixes.

**Expected Results:** The GPU name (e.g. `Tesla T4`) and `✅ Drive mounted`.

**Parameters:** none

In [ ]:
import os, shutil, subprocess
# 1) Phase 9 needs a CUDA GPU (Qwen2.5-3B + E5). Check before spending time on Drive.
if shutil.which("nvidia-smi") is None:
    print("⚠️ No NVIDIA GPU on this runtime (TPU/CPU). Runtime ▸ Change runtime type ▸ T4 / L4 / A100 GPU, "
          "then re-run from here (unless you set USE_FAKE=True for a dry run).")
else:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip())

# 2) Mount Drive — with fallbacks for the Chrome/Colab "credential propagation was unsuccessful" bug
from google.colab import drive
MYDRIVE = "/content/drive/MyDrive"
if os.path.isdir(MYDRIVE):
    print("Drive already mounted (e.g. via the Files sidebar) — skipping mount")
else:
    errors = []
    try:
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        errors.append(f"drive.mount: {e}")
        try:   # fallback: authenticate the runtime first, then mount (uses a different sign-in flow)
            from google.colab import auth
            auth.authenticate_user()
            drive.mount("/content/drive", force_remount=True)
        except Exception as e2:
            errors.append(f"auth.authenticate_user + drive.mount: {e2}")
    if not os.path.isdir(MYDRIVE):
        print("❌ Drive mount failed:", *errors, sep="\n  ")
        print("Known Colab/Chrome sign-in bug. Workarounds, easiest first:\n"
              "  1. Files sidebar (📁) ▸ 'Mount Drive' icon, then re-run this cell (it will detect the mount)\n"
              "  2. Open the notebook in Firefox or Edge (not Chrome) and run this cell again\n"
              "  3. Chrome Incognito, signed in only to the account that owns the ShopTalk folder")
        raise RuntimeError("Google Drive not mounted")
print("✅ Drive mounted")

### 📘 1.2 Configuration (single config cell)

**Purpose:** All Phase 9 parameters in one place. Overrides default to `None` = auto-detect.

**Expected Results:** No output; variables defined.

**Parameters:** paths · API · eval set · retrieval comparison · rating · latency · `max_new_tokens` sweep — see inline comments.

In [ ]:
# ===================== Phase 9 configuration — the only cell you should need to edit =====================
from pathlib import Path
import time

# --- Paths
DRIVE_PROJECT  = Path("/content/drive/MyDrive/IK classes/30_week 04Oct2026-LLM Ops - AI System Design- Sayan/ShopTalk")
REPO_LINK      = Path("/content/ShopTalk")        # symlink to DRIVE_PROJECT (no spaces → safe for %%writefile)
LOCAL_RUN      = Path("/content/shoptalk_run")    # local-disk working copy the API runs from (Chroma copied here)
RAG_CONFIG_REL = "src/rag/rag_config.json"
RESULTS_DIR    = REPO_LINK / "results" / "phase9"
RUN_ID         = time.strftime("%Y%m%d-%H%M")
EXTRA_COPY     = []          # extra repo-relative paths the API needs locally (if not referenced in rag_config)
COPY_SKIP      = ("raw", "images", "logs", "results", "notebooks")   # never copy dirs with these names
HF_HOME        = None        # e.g. "/content/drive/MyDrive/hf_cache" if Phase 7 cached Qwen on Drive

# --- API (Phase 7)
API_APP          = None      # e.g. "src.api.main:app"; None = auto-detect `= FastAPI(` under src/api
API_PORT         = 8000
USE_FAKE         = False     # True → SHOPTALK_FAKE=1 dry run of the whole notebook (numbers meaningless)
API_ENV          = {}        # extra env vars for the API process
HEALTH_TIMEOUT_S = 900
QUERY_FIELD      = "query"   # request body field for the user text
SESSION_PREFIX   = f"phase9-{RUN_ID}"
TEST_SESSION_PREFIXES = ("phase7-", "phase8-", "phase9-", "test-")   # filtered out of log mining

# --- Evaluation set (reuse Phase 6)
PHASE6_EVAL_PATH    = None   # e.g. "data/eval/eval_queries.jsonl"; None = auto-discover
RELEVANCE_THRESHOLD = 1.0    # grade ≥ this counts as relevant (graded labels)
KS                  = (1, 3, 5, 10)
PRIMARY_K           = 5
INCLUDE_LOG_CASES   = True
MAX_LOG_CASES       = 60

# --- Retrieval comparison (pretrained vs fine-tuned)
RUN_RETRIEVAL_COMPARISON = True
EMBED_BASE        = None     # None = from adapter_config.json / rag_config (e.g. "intfloat/e5-small-v2")
EMBED_ADAPTER     = None     # None = auto (LoRA dir in rag_config)
CHROMA_DIR        = None     # None = auto (dir containing chroma.sqlite3, copied to LOCAL_RUN)
CHROMA_COLLECTION = None     # None = from rag_config / the only collection
QUERY_PREFIX      = "query: "
PASSAGE_PREFIX    = None     # None = auto-detect "passage: " vs "" against the stored index embeddings
EMBED_BATCH       = 256
EXPECTED_N_PRODUCTS = 18192
REPRO_TOLERANCE   = 0.02     # |Phase 9 − Phase 6| allowed on the primary metric
PHASE6_BASELINE   = None     # manual fallback if the `evaluated` block is not recognised, e.g.
                             # {"pretrained": {"precision@5": 0.82}, "finetuned": {"precision@5": 0.87, "mrr": 0.91}}

# --- Generation quality (manual)
N_RATE        = 40           # plan: 30–50
MIN_RATINGS   = 30
RATING_SEED   = 42
QUALITY_TARGETS = {"relevance": 4.0, "groundedness": 4.0, "fluency": 4.0}   # mean on a 1–5 scale

# --- Latency
N_LATENCY          = 120     # plan: 100+
LATENCY_WARMUP     = 3
LATENCY_TARGET_MS  = 5000
N_STREAM           = 30
PHASE8_EXAMPLES    = None    # None = read the 5 example queries from src/ui
PHASE7_CLIENT_P95_MS = 4662

# --- max_new_tokens sweep
RUN_SWEEP          = True
SWEEP_VALUES       = None    # None = auto around the current value: ×1.28 (quality headroom), ×0.85, ×0.7, ×0.55
SWEEP_HEADROOM_MS  = 0       # require full-answer P95 < LATENCY_TARGET_MS − headroom (e.g. 250 for a safety margin)
SWEEP_REPEATS      = 6       # × the Phase 8 examples
SWEEP_EXTRA_EVAL_QUERIES = 10
SWEEP_TOLERANCE    = {"truncation_rate": +0.05, "citation_valid_rate": -0.02, "coverage": -0.10, "cites_any_rate": -0.05}
SPOTCHECK_N        = 10      # blind manual A/B: baseline vs recommended
APPLY_RECOMMENDED  = False   # True → write the recommended max_new_tokens into the Drive rag_config.json (backup kept)

### 📘 1.3 Install pinned dependencies

**Purpose:** Install exactly Phase 7's `requirements-api.txt` (no `-U`, no `--force-reinstall`) and verify the pins.

**Expected Results:** A version table; ✅ for chromadb 1.5.9, sentence-transformers 5.7.0, peft 0.21.2, torchao ≥ 0.16. If anything already imported changed version, the cell asks for a runtime restart.

**Parameters:** `REPO_LINK`, `DRIVE_PROJECT`.

In [ ]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError

if not REPO_LINK.exists():
    REPO_LINK.symlink_to(DRIVE_PROJECT, target_is_directory=True)
assert (REPO_LINK / RAG_CONFIG_REL).exists(), f"rag_config not found under {DRIVE_PROJECT}"

req = next((p for p in [REPO_LINK / "requirements-api.txt", REPO_LINK / "src/api/requirements-api.txt",
                        REPO_LINK / "requirements" / "requirements-api.txt"] if p.exists()), None)
assert req, "requirements-api.txt (Phase 7) not found"
before = {}
for pkg in ("chromadb", "sentence-transformers", "peft", "torchao", "transformers", "torch"):
    try: before[pkg] = version(pkg)
    except PackageNotFoundError: before[pkg] = None
print("Installing", req)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)

PINS = {"chromadb": "==1.5.9", "sentence-transformers": "==5.7.0", "peft": "==0.21.2", "torchao": ">=0.16"}
def _ok(v, spec):
    if v is None: return False
    if spec.startswith("=="): return v == spec[2:]
    want = tuple(int(x) for x in spec[2:].split("."))
    have = tuple(int(x) for x in v.split("+")[0].split(".")[:len(want)] if x.isdigit())
    return have >= want
restart = False
for pkg, spec in PINS.items():
    try: v = version(pkg)
    except PackageNotFoundError: v = None
    print(f"{'✅' if _ok(v, spec) else '❌'} {pkg:22s} {v}  (want {spec})")
    if before.get(pkg) and before[pkg] != v and pkg.replace("-", "_") in {m.split('.')[0] for m in sys.modules}:
        restart = True
for pkg in ("transformers", "torch", "fastapi", "uvicorn", "requests", "ipywidgets", "psutil"):
    try: print(f"   {pkg:22s} {version(pkg)}")
    except PackageNotFoundError: print(f"   {pkg:22s} MISSING")
if restart:
    print("\n⚠️ A package that was already imported changed version → Runtime ▸ Restart session, then re-run from 1.1.")

### 📘 1.4 Helpers & CHECKS registry

**Purpose:** Shared helpers, the `CHECKS` dict for the exit checklist, and the Phase 6 `rag_config.json` (read from Drive).

**Expected Results:** Prints the top-level keys of rag_config and whether it has an `evaluated` block.

**Parameters:** `RESULTS_DIR`.

In [ ]:
import os, sys, subprocess, json, re, math, random, shutil, ast, importlib
import numpy as np, pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKS = {}
_ICON = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "PENDING": "⏳", "SKIP": "⏭️"}
def check(name, status, detail=""):
    CHECKS[name] = {"status": status, "detail": str(detail)}
    print(f"{_ICON.get(status, '•')} {name}: {status} — {detail}")
    (RESULTS_DIR / "checks_progress.json").write_text(json.dumps(CHECKS, indent=2))

def save_json(obj, name):
    p = RESULTS_DIR / name
    p.write_text(json.dumps(obj, indent=2, default=str)); return p

def string_leaves(o, path=()):
    if isinstance(o, dict):
        for k, v in o.items(): yield from string_leaves(v, path + (k,))
    elif isinstance(o, list):
        for i, v in enumerate(o): yield from string_leaves(v, path + (i,))
    elif isinstance(o, str):
        yield path, o

def find_key_paths(o, key, path=()):
    out = []
    if isinstance(o, dict):
        for k, v in o.items():
            if k == key: out.append(path + (k,))
            out += find_key_paths(v, key, path + (k,))
    elif isinstance(o, list):
        for i, v in enumerate(o): out += find_key_paths(v, key, path + (i,))
    return out

def get_path(o, path):
    for p in path: o = o[p]
    return o

def set_path(o, path, value):
    for p in path[:-1]: o = o[p]
    o[path[-1]] = value

RAG_CFG = json.loads((REPO_LINK / RAG_CONFIG_REL).read_text())
print("rag_config keys:", list(RAG_CFG))
print("has 'evaluated' block:", "evaluated" in RAG_CFG)

### 📘 1.5 Local runtime copy (Chroma on local disk)

**Purpose:** Copy `src/`, the Chroma index and every path referenced by rag_config from Drive to local disk, and rewrite absolute Drive paths in the local config. The API runs from this copy, so Chroma never reads through the Drive FUSE mount and Phase 9 traffic does not touch the production logs.

**Expected Results:** A table of copied paths with sizes; `LOCAL_CHROMA_DIRS` lists ≥ 1 directory containing `chroma.sqlite3`.

**Parameters:** `LOCAL_RUN`, `EXTRA_COPY`, `CHROMA_DIR`.

In [ ]:
def du_mb(p):
    p = Path(p)
    if p.is_file(): return p.stat().st_size / 1e6
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6

def find_chroma_dirs(root, max_depth=4, skip=("results", "logs", ".git", "images", "raw", "notebooks")):
    root, found = Path(root), []
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")] if depth < max_depth else []
        if "chroma.sqlite3" in filenames:
            found.append(Path(dirpath).relative_to(root).as_posix()); dirnames[:] = []
    return found

def cfg_rel_paths(cfg):
    rels = set()
    for _, s in string_leaves(cfg):
        if len(s) > 300 or "\n" in s or not s.strip(): continue
        p = Path(s)
        if p.is_absolute():
            for base in (DRIVE_PROJECT, REPO_LINK):
                try: rel = p.relative_to(base).as_posix(); break
                except ValueError: rel = None
        else:
            rel = s[2:] if s.startswith("./") else s
        if (rel and rel not in (".", "", "src") and not rel.startswith("src/")
                and not any(part in COPY_SKIP for part in Path(rel).parts) and (DRIVE_PROJECT / rel).exists()):
            rels.add(rel)
    return sorted(rels)

LOCAL_RUN.mkdir(parents=True, exist_ok=True)
to_copy = ["src"] + [p.name for p in DRIVE_PROJECT.glob("requirements*.txt")]
chroma_rel = [CHROMA_DIR] if CHROMA_DIR else find_chroma_dirs(DRIVE_PROJECT)
to_copy += cfg_rel_paths(RAG_CFG) + chroma_rel + list(EXTRA_COPY)
copied = []
for rel in dict.fromkeys(to_copy):
    src, dst = DRIVE_PROJECT / rel, LOCAL_RUN / rel
    if not src.exists(): print("⚠️ missing:", rel); continue
    t0 = time.time()
    if src.is_dir():
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
    else:
        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst)
    copied.append({"path": rel, "MB": round(du_mb(dst), 1), "secs": round(time.time() - t0, 1)})
display(pd.DataFrame(copied))

# local config: absolute Drive paths → local copy
LOCAL_CFG_PATH = LOCAL_RUN / RAG_CONFIG_REL
local_cfg = json.loads(LOCAL_CFG_PATH.read_text())
for path, s in list(string_leaves(local_cfg)):
    for base in (str(DRIVE_PROJECT), str(REPO_LINK)):
        if s.startswith(base):
            set_path(local_cfg, path, str(LOCAL_RUN) + s[len(base):]); print("rewrote", "/".join(map(str, path)))
LOCAL_CFG_PATH.write_text(json.dumps(local_cfg, indent=2))
ORIGINAL_LOCAL_CFG_TEXT = LOCAL_CFG_PATH.read_text()

LOCAL_CHROMA_DIRS = [LOCAL_RUN / r for r in chroma_rel if (LOCAL_RUN / r / "chroma.sqlite3").exists()]
print("Local Chroma dirs:", LOCAL_CHROMA_DIRS)
if str(LOCAL_RUN) not in sys.path: sys.path.insert(0, str(LOCAL_RUN))
if HF_HOME: os.environ["HF_HOME"] = HF_HOME

### 📘 1.6 GPU check

**Purpose:** Confirm the runtime has a GPU (Qwen2.5-3B needs one unless `USE_FAKE=True`).

**Expected Results:** `nvidia-smi` output and `cuda: True`.

**Parameters:** `USE_FAKE`.

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if torch.cuda.is_available(): subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"])
elif not USE_FAKE: print("⚠️ No GPU: switch runtime or set USE_FAKE=True for a dry run.")

## 2. Evaluation package (`src/eval/`)

Each module is written to the repo on Drive with `%%writefile`, then synced into the local runtime copy in 2.9.

### 📘 2.0 Package init

**Purpose:** Create `src/eval/`.

**Expected Results:** Directory exists.

**Parameters:** none

In [ ]:
(REPO_LINK / "src" / "eval").mkdir(parents=True, exist_ok=True)
(REPO_LINK / "src" / "eval" / "__init__.py").write_text("# ShopTalk Phase 9 evaluation package\n")
print("ok")

### 📘 2.1 `metrics.py` — Phase 6-compatible eval-set loader + retrieval metrics

**Purpose:** Loads the Phase 6 query set in any common shape (jsonl/json/csv; id lists, graded dicts, attribute constraints) and computes P@k, judged-P@k, R@k, Hit@k, nDCG@k, MRR, attribute precision, paired bootstrap CIs, and canonical names for the Phase 6 `evaluated` metrics.

**Expected Results:** `Writing /content/ShopTalk/src/eval/metrics.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/metrics.py
"""Phase 9 retrieval metrics + Phase 6 eval-set loading.

The loader is tolerant of the common shapes a Phase 6 eval file can take
(.jsonl / .json / .csv; id lists, graded dicts, or attribute constraints) so the
same query set and labels can be reused without rewriting them.
"""
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd

QUERY_KEYS = ("query", "question", "text", "q", "user_query")
QID_KEYS = ("qid", "query_id", "id", "case_id")
GRADED_KEYS = ("relevance", "graded", "qrels", "labels", "judgments", "grades")
LIST_KEYS = ("relevant_ids", "relevant_product_ids", "relevant", "gold_ids",
             "positives", "expected_ids", "gold", "product_ids")
ATTR_KEYS = ("expected", "constraints", "expected_attributes", "expected_filters", "filters")
NO_MATCH_KEYS = ("expect_no_match", "expected_no_match", "no_match", "should_be_no_match")


# ----------------------------------------------------------------------------- loading
def _parse_listish(v: Any) -> Any:
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return []
        if s[0] in "[{":
            try:
                return json.loads(s)
            except Exception:
                pass
        return [x.strip() for x in re.split(r"[;|,]", s) if x.strip()]
    return v


def load_records(path: str | Path) -> list[dict]:
    p = Path(path)
    if p.suffix == ".jsonl":
        out = []
        for line in p.read_text().splitlines():
            line = line.strip()
            if line:
                out.append(json.loads(line))
        return out
    if p.suffix == ".json":
        data = json.loads(p.read_text())
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for k in ("queries", "items", "data", "eval", "examples", "cases", "test"):
                if isinstance(data.get(k), list):
                    return data[k]
            # dict keyed by qid
            if all(isinstance(v, dict) for v in data.values()):
                return [{"qid": k, **v} for k, v in data.items()]
        raise ValueError(f"Unrecognised JSON layout in {p}")
    if p.suffix in (".csv", ".tsv"):
        df = pd.read_csv(p, sep="\t" if p.suffix == ".tsv" else ",")
        recs = df.to_dict(orient="records")
        for r in recs:
            for k in list(r):
                if k in GRADED_KEYS + LIST_KEYS + ATTR_KEYS:
                    r[k] = _parse_listish(r[k])
        return recs
    raise ValueError(f"Unsupported eval file type: {p.suffix}")


def _qrels_from(rec: dict) -> dict[str, float]:
    for k in GRADED_KEYS:
        v = _parse_listish(rec.get(k))
        if isinstance(v, dict) and v:
            return {str(pid): float(g) for pid, g in v.items()}
        if isinstance(v, list) and v and isinstance(v[0], dict):
            out = {}
            for d in v:
                pid = d.get("product_id", d.get("id", d.get("pid")))
                g = d.get("grade", d.get("relevance", d.get("label", 1)))
                if pid is not None:
                    out[str(pid)] = float(g)
            if out:
                return out
    for k in LIST_KEYS:
        v = _parse_listish(rec.get(k))
        if isinstance(v, (list, tuple)) and v:
            return {str(x): 1.0 for x in v}
        if isinstance(v, str) and v:
            return {v: 1.0}
    return {}


def normalize_eval_set(records: Iterable[dict]) -> list[dict]:
    """-> [{qid, query, qrels{pid: grade}, expected{attr: val}, expect_no_match, history, raw}]"""
    out = []
    for i, rec in enumerate(records):
        query = next((rec[k] for k in QUERY_KEYS if isinstance(rec.get(k), str) and rec[k].strip()), None)
        if query is None:
            continue
        qid = next((str(rec[k]) for k in QID_KEYS if rec.get(k) not in (None, "")), f"q{i:03d}")
        expected = {}
        for k in ATTR_KEYS:
            v = _parse_listish(rec.get(k))
            if isinstance(v, dict) and v:
                expected = v
                break
        enm = next((rec[k] for k in NO_MATCH_KEYS if k in rec), None)
        out.append({
            "qid": qid,
            "query": query.strip(),
            "qrels": _qrels_from(rec),
            "expected": expected,
            "expect_no_match": bool(enm) if enm is not None and not (isinstance(enm, float) and math.isnan(enm)) else None,
            "history": rec.get("history") if isinstance(rec.get("history"), list) else None,
            "raw": rec,
        })
    # make qids unique
    seen: dict[str, int] = {}
    for r in out:
        if r["qid"] in seen:
            seen[r["qid"]] += 1
            r["qid"] = f"{r['qid']}_{seen[r['qid']]}"
        else:
            seen[r["qid"]] = 0
    return out


# ----------------------------------------------------------------------------- metrics
def _is_rel(pid: str, qrels: dict, thr: float) -> bool:
    return qrels.get(str(pid), 0.0) >= thr


def precision_at_k(ranked, qrels, k, thr=1.0):
    """Standard P@k: relevant-in-top-k / k (missing slots count as non-relevant)."""
    return sum(_is_rel(p, qrels, thr) for p in ranked[:k]) / k


def judged_precision_at_k(ranked, qrels, k, thr=1.0):
    """Precision over judged items only (unlabelled products ignored) - shows label holes."""
    judged = [p for p in ranked[:k] if str(p) in qrels]
    return float("nan") if not judged else sum(_is_rel(p, qrels, thr) for p in judged) / len(judged)


def recall_at_k(ranked, qrels, k, thr=1.0):
    n_rel = sum(1 for g in qrels.values() if g >= thr)
    return float("nan") if n_rel == 0 else sum(_is_rel(p, qrels, thr) for p in ranked[:k]) / n_rel


def hit_at_k(ranked, qrels, k, thr=1.0):
    return float(any(_is_rel(p, qrels, thr) for p in ranked[:k]))


def mrr_at_k(ranked, qrels, k, thr=1.0):
    for i, p in enumerate(ranked[:k], 1):
        if _is_rel(p, qrels, thr):
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranked, qrels, k):
    dcg = sum((2 ** qrels.get(str(p), 0.0) - 1) / math.log2(i + 1) for i, p in enumerate(ranked[:k], 1))
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2 ** g - 1) / math.log2(i + 1) for i, g in enumerate(ideal, 1))
    return float("nan") if idcg == 0 else dcg / idcg


def _num(x):
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    m = re.search(r"\d+(?:[.,]\d+)?", str(x).replace(",", ""))
    return float(m.group()) if m else None


def attr_match(meta: dict, expected: dict) -> bool:
    """Does a product's metadata satisfy the expected attribute constraints?"""
    if not expected:
        return False
    meta_l = {str(k).lower(): v for k, v in (meta or {}).items()}
    for key, want in expected.items():
        k = str(key).lower()
        if want in (None, "", []):
            continue
        if k in ("max_price", "price_max", "price_lte", "budget"):
            p = _num(meta_l.get("price"))
            if p is None or p > float(want) + 1e-6:
                return False
            continue
        if k in ("min_price", "price_min", "price_gte"):
            p = _num(meta_l.get("price"))
            if p is None or p < float(want) - 1e-6:
                return False
            continue
        have = meta_l.get(k)
        if have is None:
            return False
        wants = want if isinstance(want, (list, tuple, set)) else [want]
        have_s = str(have).lower()
        if not any(str(w).lower() in have_s for w in wants):
            return False
    return True


def score_query(ranked: list, qrels: dict, ks=(1, 3, 5, 10), thr=1.0,
                metas: list[dict] | None = None, expected: dict | None = None) -> dict:
    ranked = [str(p) for p in ranked]
    row: dict[str, float] = {"n_returned": len(ranked), "has_labels": float(bool(qrels))}
    if qrels:
        for k in ks:
            row[f"precision@{k}"] = precision_at_k(ranked, qrels, k, thr)
            row[f"judged_precision@{k}"] = judged_precision_at_k(ranked, qrels, k, thr)
            row[f"recall@{k}"] = recall_at_k(ranked, qrels, k, thr)
            row[f"hit@{k}"] = hit_at_k(ranked, qrels, k, thr)
            row[f"ndcg@{k}"] = ndcg_at_k(ranked, qrels, k)
        row["mrr"] = mrr_at_k(ranked, qrels, max(ks), thr)
    if expected and metas is not None:
        for k in ks:
            top = metas[:k]
            row[f"attr_precision@{k}"] = sum(attr_match(m, expected) for m in top) / k
    return row


def score_runs(runs: dict[str, list], evalset: list[dict], ks=(1, 3, 5, 10), thr=1.0,
               metas_by_qid: dict[str, list[dict]] | None = None) -> pd.DataFrame:
    rows = []
    for e in evalset:
        if e["qid"] not in runs:
            continue
        metas = (metas_by_qid or {}).get(e["qid"])
        r = score_query(runs[e["qid"]], e["qrels"], ks, thr, metas, e.get("expected"))
        rows.append({"qid": e["qid"], "query": e["query"], **r})
    return pd.DataFrame(rows)


def mean_metrics(df: pd.DataFrame) -> dict[str, float]:
    cols = [c for c in df.columns if "@" in c or c == "mrr"]
    return {c: float(np.nanmean(df[c].astype(float))) for c in cols if df[c].notna().any()}


def paired_bootstrap(a, b, n=2000, seed=0) -> dict:
    """Mean(b - a) with a 95% CI from a paired bootstrap over queries."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    if len(a) == 0:
        return {"delta": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"), "n": 0}
    rng = np.random.default_rng(seed)
    d = b - a
    idx = rng.integers(0, len(d), size=(n, len(d)))
    boots = d[idx].mean(axis=1)
    return {"delta": float(d.mean()), "ci_low": float(np.percentile(boots, 2.5)),
            "ci_high": float(np.percentile(boots, 97.5)), "n": int(len(d))}


# ----------------------------------------------------------------------------- Phase 6 baseline helpers
def flatten(d: Any, prefix: str = "") -> dict[str, Any]:
    out: dict[str, Any] = {}
    if isinstance(d, dict):
        for k, v in d.items():
            out.update(flatten(v, f"{prefix}.{k}" if prefix else str(k)))
    else:
        out[prefix] = d
    return out


_ALIASES = [
    (r"judged[_ ]?prec(?:ision)?", "judged_precision"),
    (r"attr(?:ibute)?[_ ]?prec(?:ision)?", "attr_precision"),
    (r"precision|prec|p", "precision"),
    (r"recall|rec|r", "recall"),
    (r"ndcg", "ndcg"),
    (r"hit[_ ]?rate|hits?", "hit"),
]
_NOT_METRIC = re.compile(r"latency|_ms\b|^ms$|time|sec|tokens?|size|count|^n_|^num|top_?k|temperature|lora_|rank$|alpha|dropout|epoch|lr|batch")


def _canon_component(comp: str) -> str | None:
    c = comp.lower().replace("-", "_").strip()
    if _NOT_METRIC.search(c):
        return None
    if re.search(r"(?<![a-z])mrr(?![a-z])", c):
        return "mrr"
    for pat, canon in _ALIASES:
        # metric name not glued to other letters, then @k / _at_k / _k / k (e.g. P@5, precision_at_5_ft, ft_p5)
        m = re.search(rf"(?<![a-z])(?:{pat})\s*(?:@|_at_|_at|at_|_| )?(\d{{1,2}})(?!\d)", c)
        if m and 1 <= int(m.group(1)) <= 50:
            return f"{canon}@{int(m.group(1))}"
    return None


def canon_metric(name: str) -> str | None:
    """Canonical metric for a flattened config path, looking at every component from the leaf up:
    'retrieval.finetuned.P@5', 'precision_at_5_ft', 'p@5.lora' -> 'precision@5'; None if not a metric."""
    for comp in reversed(name.split(".")):
        m = _canon_component(comp)
        if m:
            return m
    return None


def _as_number(v):
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        m = re.fullmatch(r"\s*(-?\d+(?:\.\d+)?)\s*(%?)\s*", v)
        if m:
            return float(m.group(1)) / (100 if m.group(2) else 1)
    return None


def phase6_metrics(evaluated_block: Any) -> pd.DataFrame:
    """All numeric leaves of rag_config['evaluated'] (numbers or numeric strings), with a canonical
    metric name where one is recognised (metric=None rows are printed so nothing is silently dropped)."""
    rows = []
    for path, v in flatten(evaluated_block).items():
        if isinstance(v, list) and v and all(_as_number(x) is not None for x in v):
            continue  # per-query arrays are not summary metrics
        x = _as_number(v)
        if x is None:
            continue
        rows.append({"path": path, "value": x, "metric": canon_metric(path)})
    return pd.DataFrame(rows, columns=["path", "value", "metric"])

### 📘 2.2 `client.py` — API client (JSON + SSE)

**Purpose:** `/recommend` with client round-trip timing; `/recommend/stream` with an SSE parser that records time-to-meta, time-to-first-token and full-answer time.

**Expected Results:** `Writing /content/ShopTalk/src/eval/client.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/client.py
"""HTTP client for the Phase 7 ShopTalk API (JSON + SSE streaming), used by Phase 9."""
from __future__ import annotations

import json
import time
from typing import Iterable, Iterator

import requests


def parse_sse(lines: Iterable[str | bytes | None]) -> Iterator[tuple[str, str]]:
    """Minimal spec-compliant SSE parser -> (event, data) tuples."""
    event, data = "message", []
    for raw in lines:
        if raw is None:
            continue
        line = raw.decode("utf-8") if isinstance(raw, bytes) else raw
        line = line.rstrip("\r")
        if line == "":
            if data:
                yield event, "\n".join(data)
            event, data = "message", []
            continue
        if line.startswith(":"):
            continue
        field, _, value = line.partition(":")
        if value.startswith(" "):
            value = value[1:]
        if field == "event":
            event = value
        elif field == "data":
            data.append(value)
    if data:
        yield event, "\n".join(data)


def _token_text(data: str) -> str:
    try:
        obj = json.loads(data)
    except Exception:
        return data
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for k in ("token", "text", "delta", "content"):
            if isinstance(obj.get(k), str):
                return obj[k]
    return ""


class ShopTalkClient:
    def __init__(self, base_url: str, timeout: float = 180, query_field: str = "query",
                 extra_body: dict | None = None):
        self.base = base_url.rstrip("/")
        self.timeout = timeout
        self.query_field = query_field
        self.extra_body = dict(extra_body or {})
        self.s = requests.Session()

    # ------------------------------------------------------------------ basics
    def health(self) -> dict:
        r = self.s.get(f"{self.base}/health", timeout=10)
        r.raise_for_status()
        return r.json()

    def metrics(self) -> str:
        r = self.s.get(f"{self.base}/metrics", timeout=10)
        r.raise_for_status()
        return r.text

    def wait_healthy(self, timeout_s: float = 900, poll_s: float = 5, proc=None, verbose=True) -> dict:
        t0, last = time.time(), None
        while time.time() - t0 < timeout_s:
            if proc is not None and proc.poll() is not None:
                raise RuntimeError(f"API process exited with code {proc.returncode}")
            try:
                last = self.health()
                if str(last.get("status", "")).lower() == "ok":
                    if verbose:
                        print(f"✅ /health ok after {time.time() - t0:.0f}s: {last}")
                    return last
            except Exception as e:  # not up yet
                last = {"error": repr(e)[:120]}
            time.sleep(poll_s)
        raise TimeoutError(f"/health not ok after {timeout_s}s; last={last}")

    def body(self, query: str, session_id: str, history: list | None = None, **kw) -> dict:
        b = {**self.extra_body, self.query_field: query, "session_id": session_id}
        if history:
            b["history"] = history
        b.update(kw)
        return b

    # ------------------------------------------------------------------ endpoints
    def recommend(self, query: str, session_id: str, history: list | None = None, **kw) -> dict:
        t0 = time.perf_counter()
        r = self.s.post(f"{self.base}/recommend", json=self.body(query, session_id, history, **kw),
                        timeout=self.timeout)
        client_ms = (time.perf_counter() - t0) * 1000
        try:
            out = r.json()
        except Exception:
            out = {"error": r.text[:500]}
        if not isinstance(out, dict):
            out = {"payload": out}
        out["_client_ms"] = client_ms
        out["_http_status"] = r.status_code
        return out

    def recommend_stream(self, query: str, session_id: str, history: list | None = None, **kw) -> dict:
        t0 = time.perf_counter()
        meta, done, err = None, None, None
        ttft = meta_ms = None
        tokens: list[str] = []
        with self.s.post(f"{self.base}/recommend/stream", json=self.body(query, session_id, history, **kw),
                         stream=True, timeout=self.timeout,
                         headers={"Accept": "text/event-stream"}) as r:
            status = r.status_code
            for ev, data in parse_sse(r.iter_lines(decode_unicode=True)):
                now = (time.perf_counter() - t0) * 1000
                if ev == "meta":
                    meta_ms = now
                    try:
                        meta = json.loads(data)
                    except Exception:
                        meta = {"raw": data}
                elif ev == "token":
                    if ttft is None:
                        ttft = now
                    tokens.append(_token_text(data))
                elif ev == "done":
                    try:
                        done = json.loads(data)
                    except Exception:
                        done = {"raw": data}
                    break
                elif ev == "error":
                    err = data
                    break
        total = (time.perf_counter() - t0) * 1000
        streamed = "".join(tokens)
        answer = (done or {}).get("answer") if isinstance(done, dict) else None
        return {
            "http_status": status, "ttft_ms": ttft, "meta_ms": meta_ms, "total_ms": total,
            "answer": answer if isinstance(answer, str) and answer else streamed,
            "n_token_events": len(tokens), "meta": meta, "done": done, "error": err,
        }

### 📘 2.3 `api_server.py` — API subprocess manager

**Purpose:** Detects the FastAPI app module, frees the port, starts uvicorn as a subprocess with its own log, stops it cleanly.

**Expected Results:** `Writing /content/ShopTalk/src/eval/api_server.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/api_server.py
"""Start / stop the Phase 7 FastAPI service as a subprocess (Colab-friendly)."""
from __future__ import annotations

import os
import re
import signal
import subprocess
import sys
import time
from pathlib import Path


def detect_app_module(repo: str | Path, pkg: str = "src/api") -> tuple[str, bool]:
    """Find `<var> = FastAPI(` (or a `create_app` factory) under src/api.
    Returns ("src.api.main:app", is_factory)."""
    repo = Path(repo)
    files = sorted((repo / pkg).glob("*.py"),
                   key=lambda p: (p.stem not in ("main", "app", "server", "api"), p.stem))
    dotted = pkg.strip("/").replace("/", ".")
    for f in files:
        txt = f.read_text(errors="ignore")
        m = re.search(r"^(\w+)\s*(?::\s*\w+\s*)?=\s*FastAPI\(", txt, re.M)
        if m:
            return f"{dotted}.{f.stem}:{m.group(1)}", False
    for f in files:
        if re.search(r"^def create_app\(", f.read_text(errors="ignore"), re.M):
            return f"{dotted}.{f.stem}:create_app", True
    raise FileNotFoundError(f"No FastAPI app found under {repo / pkg}; set API_APP in the config cell")


def free_port(port: int) -> None:
    """Kill whatever is listening on `port` (left over from a previous run)."""
    try:
        import psutil
        for c in psutil.net_connections(kind="inet"):
            if c.laddr and c.laddr.port == port and c.status == psutil.CONN_LISTEN and c.pid:
                try:
                    os.kill(c.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
        time.sleep(1)
    except Exception:
        subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)


class ApiServer:
    def __init__(self, repo: str | Path, app: str, port: int = 8000, env: dict | None = None,
                 log_path: str | Path | None = None, factory: bool = False):
        self.repo, self.app, self.port, self.factory = Path(repo), app, port, factory
        self.env = {**os.environ, **{k: str(v) for k, v in (env or {}).items()}}
        self.env["PYTHONPATH"] = f"{self.repo}:{self.env.get('PYTHONPATH', '')}"
        self.log_path = Path(log_path or self.repo / "logs" / "phase9_uvicorn.log")
        self.proc: subprocess.Popen | None = None

    @property
    def url(self) -> str:
        return f"http://127.0.0.1:{self.port}"

    def start(self) -> subprocess.Popen:
        free_port(self.port)
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        cmd = [sys.executable, "-m", "uvicorn", self.app, "--host", "127.0.0.1",
               "--port", str(self.port), "--log-level", "warning"]
        if self.factory:
            cmd.append("--factory")
        self._log = open(self.log_path, "a")
        self._log.write(f"\n===== start {time.strftime('%F %T')} : {' '.join(cmd)}\n")
        self._log.flush()
        self.proc = subprocess.Popen(cmd, cwd=self.repo, env=self.env, stdout=self._log,
                                     stderr=subprocess.STDOUT, start_new_session=True)
        return self.proc

    def stop(self, timeout: float = 30) -> None:
        if self.proc and self.proc.poll() is None:
            try:
                os.killpg(self.proc.pid, signal.SIGTERM)
                self.proc.wait(timeout=timeout)
            except Exception:
                try:
                    os.killpg(self.proc.pid, signal.SIGKILL)
                except Exception:
                    pass
        self.proc = None
        try:
            self._log.close()
        except Exception:
            pass

    def tail(self, n: int = 40) -> str:
        try:
            return "\n".join(self.log_path.read_text(errors="ignore").splitlines()[-n:])
        except FileNotFoundError:
            return "(no log yet)"

### 📘 2.4 `quality.py` — Automatic answer-quality proxies

**Purpose:** Citation validity (`[ID: …]` ⊆ returned products), hallucinated IDs, coverage, price grounding, truncation.

**Expected Results:** `Writing /content/ShopTalk/src/eval/quality.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/quality.py
"""Automatic answer-quality proxies (complement - not replace - the manual ratings)."""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

CITE_RE = re.compile(r"\[ID:\s*([^\]]+)\]", re.I)
PRICE_RE = re.compile(r"\$\s?(\d{1,3}(?:,\d{3})*(?:\.\d{1,2})?|\d+(?:\.\d{1,2})?)")
CLEAN_END = tuple('.!?)"\'*]`')


def cited_ids(answer: str) -> list[str]:
    out = []
    for m in CITE_RE.finditer(answer or ""):
        out += [x.strip() for x in re.split(r"[,;\s]+", m.group(1)) if x.strip()]
    return out


def _price(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    m = re.search(r"\d+(?:\.\d+)?", str(v).replace(",", ""))
    return float(m.group()) if m else None


def answer_proxies(answer: str, products: list[dict], no_match: bool = False) -> dict:
    answer = answer or ""
    returned = [str(p.get("id")) for p in products or []]
    cited = cited_ids(answer)
    cited_set, ret_set = set(cited), set(returned)
    valid = cited_set & ret_set
    prices = [float(x.replace(",", "")) for x in PRICE_RE.findall(answer)]
    prod_prices = [p for p in (_price(pr.get("price")) for pr in products or []) if p is not None]
    grounded_prices = [x for x in prices if any(abs(x - pp) < 0.011 for pp in prod_prices)]
    stripped = answer.rstrip()
    return {
        "answer_words": len(answer.split()),
        "answer_chars": len(answer),
        "empty_answer": not stripped,
        "n_returned": len(returned),
        "n_cited": len(cited_set),
        "cites_any": bool(cited_set),
        "citation_valid_rate": (len(valid) / len(cited_set)) if cited_set else np.nan,
        "hallucinated_ids": sorted(cited_set - ret_set),
        "coverage": (len(valid) / len(ret_set)) if ret_set else np.nan,
        "n_prices": len(prices),
        "price_grounded_rate": (len(grounded_prices) / len(prices)) if prices else np.nan,
        "ends_cleanly": bool(stripped) and stripped.endswith(CLEAN_END),
        "no_match": bool(no_match),
    }


def summarize(df: pd.DataFrame) -> dict:
    if df.empty:
        return {}
    has_products = df["n_returned"] > 0
    return {
        "n": int(len(df)),
        "mean_words": float(df["answer_words"].mean()),
        "p95_words": float(df["answer_words"].quantile(0.95)),
        "empty_rate": float(df["empty_answer"].mean()),
        "cites_any_rate": float(df.loc[has_products, "cites_any"].mean()) if has_products.any() else np.nan,
        "citation_valid_rate": float(np.nanmean(df["citation_valid_rate"])) if df["citation_valid_rate"].notna().any() else np.nan,
        "hallucinated_id_rate": float((df["hallucinated_ids"].map(len) > 0).mean()),
        "coverage": float(np.nanmean(df["coverage"])) if df["coverage"].notna().any() else np.nan,
        "price_grounded_rate": float(np.nanmean(df["price_grounded_rate"])) if df["price_grounded_rate"].notna().any() else np.nan,
        "truncation_rate": float((~df["ends_cleanly"] & ~df["empty_answer"]).mean()),
        "no_match_rate": float(df["no_match"].mean()),
    }

### 📘 2.5 `logs_mining.py` — Logs → evaluation cases

**Purpose:** Reads `requests.jsonl` + `feedback.jsonl`, drops test sessions, joins on `request_id`, returns thumbs-down and no-match cases (deduplicated by query) plus click/rating statistics.

**Expected Results:** `Writing /content/ShopTalk/src/eval/logs_mining.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/logs_mining.py
"""Turn production logs (Phase 7 JSONL) into Phase 9 evaluation cases."""
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd

DEFAULT_TEST_PREFIXES = ("phase7-", "phase8-", "phase9-", "test-")
DOWN = {"down", "thumbs_down", "thumbsdown", "-1", "bad", "negative", "dislike", "👎", "no", "false"}
UP = {"up", "thumbs_up", "thumbsup", "1", "+1", "good", "positive", "like", "👍", "yes", "true"}


def read_jsonl(path: str | Path) -> tuple[list[dict], int]:
    p = Path(path)
    if not p.exists():
        return [], 0
    out, bad = [], 0
    for line in p.read_text(errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            if isinstance(rec, dict):
                out.append(rec)
            else:
                bad += 1
        except Exception:
            bad += 1
    return out, bad


def is_test_session(sid, prefixes=DEFAULT_TEST_PREFIXES) -> bool:
    return isinstance(sid, str) and sid.lower().startswith(tuple(p.lower() for p in prefixes))


def classify_feedback(rec: dict) -> str | None:
    kind = str(rec.get("kind") or rec.get("type") or rec.get("event") or "").lower()
    rating = next((rec[k] for k in ("rating", "vote", "thumbs", "value", "score") if k in rec), None)
    if "click" in kind or (rating is None and "product_id" in rec):
        return "click"
    if isinstance(rating, bool):
        return "up" if rating else "down"
    if isinstance(rating, (int, float)):
        return "up" if rating > 0 else "down" if rating < 0 else None
    if rating is not None:
        r = str(rating).strip().lower()
        if r in DOWN:
            return "down"
        if r in UP:
            return "up"
    return None


def truthy(v) -> bool:
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    try:
        if v != v:  # NaN
            return False
    except Exception:
        pass
    return bool(v)


def _norm_q(q: str) -> str:
    return re.sub(r"\s+", " ", str(q or "").strip().lower())


def _latency_total(v):
    if isinstance(v, dict):
        return v.get("total")
    return v if isinstance(v, (int, float)) else None


def mine(requests_path, feedback_path, prefixes=DEFAULT_TEST_PREFIXES) -> dict:
    reqs, bad_r = read_jsonl(requests_path)
    fbs, bad_f = read_jsonl(feedback_path)
    rq = pd.DataFrame(reqs)
    if rq.empty:
        rq = pd.DataFrame(columns=["request_id", "query", "session_id"])
    for c in ("request_id", "query", "session_id", "no_match", "product_ids", "answer", "latency"):
        if c not in rq.columns:
            rq[c] = None
    rq["is_test"] = rq["session_id"].map(lambda s: is_test_session(s, prefixes)).astype(bool)
    real = rq[~rq["is_test"]].copy()
    sess_by_req = dict(zip(rq["request_id"], rq["session_id"]))

    fb = pd.DataFrame(fbs)
    if fb.empty:
        fb = pd.DataFrame(columns=["request_id"])
    fb["fb_type"] = [classify_feedback(r) for r in fbs] if fbs else []
    if "session_id" not in fb.columns:
        fb["session_id"] = None
    fb["session_id"] = [s if isinstance(s, str) else sess_by_req.get(r)
                        for s, r in zip(fb["session_id"], fb["request_id"])]
    fb["is_test"] = fb["session_id"].map(lambda s: is_test_session(s, prefixes)).astype(bool)
    fb_real = fb[~fb["is_test"]].copy()

    comment_col = next((c for c in ("comment", "text", "note") if c in fb_real.columns), None)
    downs = fb_real[fb_real["fb_type"] == "down"]
    down_comments = downs.groupby("request_id")[comment_col].apply(
        lambda s: " | ".join(str(x) for x in s if isinstance(x, str) and x.strip())) if comment_col and not downs.empty else pd.Series(dtype=str)

    cases = []
    down_ids = set(downs["request_id"])
    for _, r in real.iterrows():
        reasons = []
        if r["request_id"] in down_ids:
            reasons.append("thumbs_down")
        if truthy(r.get("no_match")):
            reasons.append("no_match")
        if not reasons or not isinstance(r["query"], str) or not r["query"].strip():
            continue
        cases.append({
            "request_id": r["request_id"], "session_id": r["session_id"], "query": r["query"].strip(),
            "reason": "+".join(reasons), "comment": down_comments.get(r["request_id"], ""),
            "logged_product_ids": r["product_ids"] if isinstance(r["product_ids"], list) else [],
            "logged_no_match": truthy(r.get("no_match")), "logged_answer": r["answer"],
            "logged_latency_total": _latency_total(r["latency"]), "ts": r.get("ts", r.get("timestamp")),
        })
    cases_df = pd.DataFrame(cases)
    n_before_dedup = len(cases_df)
    if not cases_df.empty:
        cases_df["qnorm"] = cases_df["query"].map(_norm_q)
        agg_reason = cases_df.groupby("qnorm")["reason"].apply(lambda s: "+".join(sorted(set("+".join(s).split("+")))))
        cases_df = cases_df.drop_duplicates("qnorm", keep="last").copy()
        cases_df["reason"] = cases_df["qnorm"].map(agg_reason)
        cases_df = cases_df.drop(columns="qnorm").reset_index(drop=True)
        cases_df.insert(0, "case_id", [f"log{i:03d}" for i in range(len(cases_df))])

    clicks = fb_real[fb_real["fb_type"] == "click"]
    n_real = len(real)
    stats = {
        "requests_total": int(len(rq)), "requests_test_filtered": int(rq["is_test"].sum()),
        "requests_real": int(n_real), "bad_lines": int(bad_r + bad_f),
        "feedback_total": int(len(fb)), "feedback_test_filtered": int(fb["is_test"].sum()) if len(fb) else 0,
        "thumbs_up": int((fb_real["fb_type"] == "up").sum()), "thumbs_down": int((fb_real["fb_type"] == "down").sum()),
        "clicks": int(len(clicks)),
        "ctr": float(clicks["request_id"].nunique() / n_real) if n_real else float("nan"),
        "mean_click_rank": float(pd.to_numeric(clicks.get("rank"), errors="coerce").mean()) if len(clicks) else float("nan"),
        "no_match_rate": float(real["no_match"].map(truthy).mean()) if n_real else float("nan"),
        "cases_before_dedup": int(n_before_dedup), "cases": int(len(cases_df)),
    }
    return {"cases": cases_df, "stats": stats, "requests_real": real, "feedback_real": fb_real}

### 📘 2.6 `latency.py` — Latency harness

**Purpose:** Runs N requests (warm-up excluded) and builds P50/P95/P99 tables.

**Expected Results:** `Writing /content/ShopTalk/src/eval/latency.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/latency.py
"""Latency harness + percentile tables for Phase 9 (reused in Phase 10 for speech-to-text)."""
from __future__ import annotations

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Callable

import numpy as np
import pandas as pd


def percentiles(values, ps=(50, 95, 99)) -> dict:
    v = np.asarray([x for x in values if x is not None and not np.isnan(x)], float)
    if v.size == 0:
        return {"n": 0, **{f"p{p}": np.nan for p in ps}, "mean": np.nan, "max": np.nan}
    return {"n": int(v.size), "mean": float(v.mean()), **{f"p{p}": float(np.percentile(v, p)) for p in ps},
            "max": float(v.max())}


def latency_table(df: pd.DataFrame, cols: dict[str, str], ps=(50, 95, 99)) -> pd.DataFrame:
    """cols = {label: column_name}"""
    rows = []
    for label, col in cols.items():
        if col in df.columns:
            rows.append({"stage": label, **percentiles(pd.to_numeric(df[col], errors="coerce"), ps)})
    out = pd.DataFrame(rows)
    return out.round(1)


def run_requests(fn: Callable[[str, int], dict], queries: list[str], n: int, warmup: int = 3,
                 concurrency: int = 1, desc: str = "requests") -> list[dict]:
    """Call fn(query, i) n times (cycling through queries) after `warmup` unrecorded calls."""
    try:
        from tqdm.auto import tqdm
    except Exception:  # pragma: no cover
        tqdm = lambda x, **k: x  # noqa: E731
    for i in range(warmup):
        try:
            fn(queries[i % len(queries)], -1 - i)
        except Exception:
            pass
    jobs = [(i, queries[i % len(queries)]) for i in range(n)]
    results: list[dict] = []

    def _one(i, q):
        t0 = time.time()
        try:
            r = fn(q, i)
            return {"i": i, "query": q, "ok": True, "started": t0, **r}
        except Exception as e:
            return {"i": i, "query": q, "ok": False, "error": repr(e)[:300], "started": t0}

    if concurrency <= 1:
        for i, q in tqdm(jobs, desc=desc):
            results.append(_one(i, q))
    else:
        with ThreadPoolExecutor(concurrency) as ex:
            futs = [ex.submit(_one, i, q) for i, q in jobs]
            for f in tqdm(as_completed(futs), total=len(futs), desc=desc):
                results.append(f.result())
    return sorted(results, key=lambda r: r["i"])

### 📘 2.7 `failures.py` — Failure taxonomy + write-up

**Purpose:** Tags each case with failure categories and renders `failure_cases.md` with a ranked improvement plan.

**Expected Results:** `Writing /content/ShopTalk/src/eval/failures.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/failures.py
"""Failure taxonomy + markdown write-up for Phase 9."""
from __future__ import annotations

import math
from datetime import datetime

import pandas as pd

CATEGORIES = {
    "retrieval_miss": {
        "desc": "Labelled query where no relevant product reached the top-k.",
        "fix": "Mine these queries as hard negatives/positives for the next E5 LoRA round; add a sparse "
               "(BM25) leg for brand / model-number / size tokens and fuse with the dense scores (RRF).",
        "measure": "precision@5 / hit@5 on the Phase 6 set + these cases",
        "severity": 3},
    "low_attr_precision": {
        "desc": "Fewer than half of the top-5 products satisfy the query's stated attributes.",
        "fix": "Tighten the query parser (colour/type/brand synonyms, price phrases) and apply parsed filters "
               "before ranking; fall back progressively (drop colour, then brand) instead of dropping all.",
        "measure": "attr_precision@5",
        "severity": 2},
    "false_no_match": {
        "desc": "System said 'no match' although the query is answerable from the catalog.",
        "fix": "Relax filter cascade before declaring no-match; log which constraint emptied the result set; "
               "normalise colour/product_type vocab to the ABO values.",
        "measure": "no-match accuracy on queries labelled expect_no_match=False",
        "severity": 3},
    "missed_no_match": {
        "desc": "Query labelled as unanswerable, but the system recommended products anyway.",
        "fix": "Add a relevance-score floor + constraint check before generation; answer honestly when nothing fits.",
        "measure": "no-match accuracy on queries labelled expect_no_match=True",
        "severity": 2},
    "unmet_constraints": {
        "desc": "Products returned while some parsed constraints were not met.",
        "fix": "State the relaxed constraint explicitly in the answer and offer the nearest alternatives.",
        "measure": "share of answers that mention every unmet constraint",
        "severity": 1},
    "hallucinated_citation": {
        "desc": "Answer cites an [ID: ...] that is not among the returned products.",
        "fix": "Post-validate citations against the retrieved IDs (strip/repair) and keep the IDs in the prompt "
               "as a closed list; penalise in the prompt with an explicit instruction.",
        "measure": "hallucinated_id_rate (target 0)",
        "severity": 3},
    "uncited_answer": {
        "desc": "Products were returned but the answer cites none of them.",
        "fix": "Few-shot example with citations in the system prompt; validator that appends IDs if missing.",
        "measure": "cites_any_rate",
        "severity": 1},
    "truncated_answer": {
        "desc": "Answer stops mid-sentence (hit max_new_tokens).",
        "fix": "Ask for a fixed, short format (e.g. 3 bullets, 1 line each); trim at the last full sentence; "
               "set max_new_tokens from the Phase 9 sweep.",
        "measure": "truncation_rate",
        "severity": 2},
    "slow": {
        "desc": "Client round-trip above the latency target.",
        "fix": "Lower max_new_tokens, shorten the prompt (fewer/shorter product cards), 4-bit/AWQ weights or vLLM; "
               "keep streaming so first token stays < 1 s.",
        "measure": "P95 total / full-answer latency",
        "severity": 2},
    "low_relevance_rating": {"desc": "Manual relevance rating ≤ 2.", "fix": "See retrieval fixes; inspect query type.",
                             "measure": "mean relevance rating", "severity": 3},
    "low_groundedness_rating": {"desc": "Manual groundedness rating ≤ 2.",
                                "fix": "Restrict the answer to fields present in the product cards; lower temperature.",
                                "measure": "mean groundedness rating", "severity": 3},
    "low_fluency_rating": {"desc": "Manual fluency rating ≤ 2.", "fix": "Prompt for structure; check repetition penalty.",
                           "measure": "mean fluency rating", "severity": 1},
    "user_thumbs_down": {"desc": "Real user gave a thumbs-down (from feedback.jsonl).",
                         "fix": "Triage the comment; route into the categories above.", "measure": "thumbs-down rate",
                         "severity": 2},
}


def _f(x):
    try:
        x = float(x)
        return None if math.isnan(x) else x
    except Exception:
        return None


def classify(row: dict, k: int = 5, slow_ms: float = 5000) -> list[str]:
    tags = []
    hit = _f(row.get(f"hit@{k}"))
    if row.get("has_labels") and hit == 0.0:
        tags.append("retrieval_miss")
    ap = _f(row.get(f"attr_precision@{k}"))
    if ap is not None and ap < 0.5 and not row.get("no_match"):
        tags.append("low_attr_precision")
    enm = row.get("expect_no_match")
    if enm is False and row.get("no_match"):
        tags.append("false_no_match")
    if enm is True and not row.get("no_match"):
        tags.append("missed_no_match")
    if row.get("unmet_constraints") and (row.get("n_returned") or 0) > 0:
        tags.append("unmet_constraints")
    if row.get("hallucinated_ids"):
        tags.append("hallucinated_citation")
    if (row.get("n_returned") or 0) > 0 and not row.get("cites_any") and not row.get("no_match"):
        tags.append("uncited_answer")
    if not row.get("ends_cleanly", True) and not row.get("empty_answer"):
        tags.append("truncated_answer")
    if (_f(row.get("client_ms")) or 0) > slow_ms:
        tags.append("slow")
    for dim in ("relevance", "groundedness", "fluency"):
        v = _f(row.get(dim))
        if v is not None and v <= 2:
            tags.append(f"low_{dim}_rating")
    if "thumbs_down" in str(row.get("reason") or ""):
        tags.append("user_thumbs_down")
    return tags


def _md_table(df: pd.DataFrame) -> str:
    if df is None or df.empty:
        return "_(none)_\n"
    df = df.copy()
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, float):
                v = f"{v:.3f}" if abs(v) < 100 else f"{v:,.0f}"
            cells.append(str(v).replace("|", "\\|").replace("\n", " "))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines) + "\n"


def build_report(cases: pd.DataFrame, headline: dict, sections: dict[str, pd.DataFrame],
                 n_examples: int = 3, title: str = "ShopTalk Phase 9 — Failure cases & improvement plan") -> str:
    """cases needs columns: case_id, source, query, tags(list), answer, top_titles."""
    md = [f"# {title}", f"_Generated {datetime.now():%Y-%m-%d %H:%M}_\n", "## 1. Headline results\n"]
    md.append(_md_table(pd.DataFrame([{"metric": k, "value": v} for k, v in headline.items()])))
    for name, df in sections.items():
        md += [f"\n### {name}\n", _md_table(df)]

    exploded = cases.explode("tags").dropna(subset=["tags"])
    counts = exploded.groupby("tags")["case_id"].nunique().rename("cases").reset_index()
    counts["share"] = counts["cases"] / max(len(cases), 1)
    counts["severity"] = counts["tags"].map(lambda t: CATEGORIES.get(t, {}).get("severity", 1))
    counts["priority_score"] = counts["cases"] * counts["severity"]
    counts = counts.sort_values("priority_score", ascending=False)

    md += ["\n## 2. Failure taxonomy\n", f"{len(cases)} evaluated cases; "
           f"{cases['tags'].map(bool).sum()} have at least one failure tag.\n",
           _md_table(counts.rename(columns={"tags": "category"}))]

    md.append("\n## 3. Examples per category\n")
    for cat in counts["tags"]:
        info = CATEGORIES.get(cat, {})
        md.append(f"\n### {cat}\n_{info.get('desc', '')}_\n")
        ex = cases[cases["tags"].map(lambda t: cat in t)].head(n_examples)
        for _, r in ex.iterrows():
            ans = str(r.get("answer") or "").replace("\n", " ")
            md.append(f"- **{r['case_id']}** ({r.get('source', '')}) — “{r['query']}”  \n"
                      f"  top products: {r.get('top_titles', '')}  \n"
                      f"  answer: {ans[:300]}{'…' if len(ans) > 300 else ''}  \n"
                      f"  tags: {', '.join(r['tags'])}")
    md.append("\n## 4. Improvement plan (ranked by cases × severity)\n")
    plan = [{"#": i + 1, "category": c, "cases": int(n), "proposed fix": CATEGORIES.get(c, {}).get("fix", ""),
             "how we measure": CATEGORIES.get(c, {}).get("measure", "")}
            for i, (c, n) in enumerate(zip(counts["tags"], counts["cases"]))]
    md.append(_md_table(pd.DataFrame(plan)))
    md.append("\n> Review and edit this plan: the fixes are starting points generated from the failure "
              "categories; confirm each against the examples above before scheduling the work.\n")
    return "\n".join(md)

### 📘 2.8 `rater.py` — Manual rating widget

**Purpose:** ipywidgets rater for relevance / groundedness / fluency (1–5) that saves to CSV after every click.

**Expected Results:** `Writing /content/ShopTalk/src/eval/rater.py`.

**Parameters:** none

In [ ]:
%%writefile /content/ShopTalk/src/eval/rater.py
"""In-notebook manual rating widget (ipywidgets). Saves to CSV after every click, so it resumes."""
from __future__ import annotations

import html
from pathlib import Path

import pandas as pd

DIMS = ("relevance", "groundedness", "fluency")
RUBRIC = {
    "relevance": "5 = products & answer fit every stated need · 3 = partly · 1 = unrelated",
    "groundedness": "5 = every claim (price, colour, brand, features, IDs) is in the product cards · 1 = invented",
    "fluency": "5 = clear, natural, well organised · 3 = understandable but clumsy · 1 = broken",
}


def make_rater(sheet_path: str | Path, dims=DIMS):
    import ipywidgets as w
    from IPython.display import display

    sheet_path = Path(sheet_path)
    df = pd.read_csv(sheet_path)
    for d in dims:
        if d not in df.columns:
            df[d] = pd.NA
    if "notes" not in df.columns:
        df["notes"] = ""
    df["notes"] = df["notes"].fillna("").astype(str)

    def todo_idx():
        return [i for i in df.index if df.loc[i, list(dims)].isna().any()]

    state = {"i": (todo_idx() or [0])[0]}
    view = w.HTML()
    progress = w.HTML()
    toggles = {d: w.ToggleButtons(options=[1, 2, 3, 4, 5], value=None, description=d[:12],
                                  tooltips=[RUBRIC[d]] * 5, style={"button_width": "42px"}) for d in dims}
    notes = w.Text(placeholder="notes (optional)", layout=w.Layout(width="70%"))
    b_prev, b_save, b_skip = w.Button(description="◀ Prev"), w.Button(description="Save & next ▶",
                                                                        button_style="success"), w.Button(description="Skip")

    def render():
        i = state["i"]
        r = df.loc[i]
        prods = html.escape(str(r.get("products_text", ""))).replace("\n", "<br>")
        ans = html.escape(str(r.get("answer", ""))).replace("\n", "<br>")
        view.value = (f"<div style='font-family:sans-serif;max-width:900px'>"
                      f"<b>{html.escape(str(r.get('case_id', i)))}</b> &nbsp;<i>{html.escape(str(r.get('source', '')))}</i>"
                      f"<h4 style='margin:6px 0'>Q: {html.escape(str(r['query']))}</h4>"
                      f"<div style='background:#f6f6f6;padding:6px;font-size:12px'>{prods}</div>"
                      f"<div style='border-left:3px solid #4a7;padding:6px;margin-top:6px'>{ans}</div></div>")
        for d in dims:
            v = r[d]
            toggles[d].value = int(v) if pd.notna(v) else None
        notes.value = str(r.get("notes", "") or "")
        done = len(df) - len(todo_idx())
        progress.value = f"<b>{done}/{len(df)} rated</b> · row {i + 1} · rubric: " + " | ".join(
            f"<i>{d}</i>: {RUBRIC[d]}" for d in dims)

    def save(_=None):
        i = state["i"]
        for d in dims:
            if toggles[d].value is not None:
                df.loc[i, d] = toggles[d].value
        df.loc[i, "notes"] = notes.value
        df.to_csv(sheet_path, index=False)

    def go(step):
        state["i"] = (state["i"] + step) % len(df)
        render()

    def on_save(_):
        save()
        rest = todo_idx()
        if rest:
            later = [j for j in rest if j > state["i"]]
            state["i"] = (later or rest)[0]
            render()
        else:
            render()
            progress.value = "<b style='color:green'>All rows rated ✔ — saved to " + html.escape(str(sheet_path)) + "</b>"

    b_save.on_click(on_save)
    b_skip.on_click(lambda _: go(1))
    b_prev.on_click(lambda _: go(-1))
    render()
    display(w.VBox([progress, view, *toggles.values(), notes, w.HBox([b_prev, b_skip, b_save])]))
    return df

### 📘 2.9 Sync to the local copy, import, self-test

**Purpose:** Copy `src/eval` into `LOCAL_RUN`, import it, and run a few assertions on toy data.

**Expected Results:** `self-test ok`.

**Parameters:** none

In [ ]:
shutil.copytree(REPO_LINK / "src" / "eval", LOCAL_RUN / "src" / "eval", dirs_exist_ok=True)
import src.eval.metrics as M, src.eval.client as C, src.eval.quality as Q, src.eval.logs_mining as L
import src.eval.latency as LT, src.eval.failures as F, src.eval.api_server as A, src.eval.rater as R
for m in (M, C, Q, L, LT, F, A, R): importlib.reload(m)

assert M.precision_at_k(["a", "x", "b"], {"a": 1, "b": 1}, 5) == 0.4
assert abs(M.ndcg_at_k(["a"], {"a": 1}, 5) - 1) < 1e-9
assert M.canon_metric("finetuned.P@5") == "precision@5" and M.canon_metric("n_queries") is None
assert list(C.parse_sse(["event: token", "data: hi", ""])) == [("token", "hi")]
assert Q.answer_proxies("x [ID: B1].", [{"id": "B1"}])["citation_valid_rate"] == 1.0
assert L.classify_feedback({"rating": "down"}) == "down" and L.is_test_session("phase8-x")
print("self-test ok")

## 3. Evaluation set

### 📘 3.1 Phase 6 baseline (`evaluated` block)

**Purpose:** Read the Phase 6 numbers we compare against: the `evaluated` block in rag_config plus any metrics JSON in `results/phase6/`. Each numeric leaf gets a canonical metric name and a model tag (pretrained / finetuned).

**Expected Results:** The raw `evaluated` block, then a table `source · path · metric · model · value`. `phase6_baseline_found` = PASS when at least one recognised metric exists.

**Parameters:** none

In [ ]:
EVALUATED = RAG_CFG.get("evaluated", {})
print(json.dumps(EVALUATED, indent=2, default=str)[:4000])

frames = []
p6 = M.phase6_metrics(EVALUATED); p6.insert(0, "source", "rag_config.evaluated"); frames.append(p6)
for f in sorted((REPO_LINK / "results" / "phase6").glob("*.json")) if (REPO_LINK / "results" / "phase6").exists() else []:
    try:
        d = M.phase6_metrics(json.loads(f.read_text())); d.insert(0, "source", f"results/phase6/{f.name}"); frames.append(d)
    except Exception as e:
        print("skip", f.name, e)
if PHASE6_BASELINE:
    frames.append(pd.DataFrame([{"source": "config PHASE6_BASELINE", "path": f"{model}.{m}", "value": float(v), "metric": M.canon_metric(m) or m}
                                for model, d in PHASE6_BASELINE.items() for m, v in d.items()]))
P6 = pd.concat(frames, ignore_index=True)
def model_tag(path):
    p = path.lower()
    if re.search(r"pre[_-]?train|base(line)?|vanilla|zero[_-]?shot|untuned", p): return "pretrained"
    if re.search(r"fine|lora|tuned|\bft\b|_ft|ft_|adapter", p): return "finetuned"
    return "finetuned?"   # Phase 6 shipped the fine-tuned model, so untagged numbers are assumed to be it
P6["model"] = P6["path"].map(model_tag)
P6_METRICS = P6.dropna(subset=["metric"]).reset_index(drop=True)
def p6_value(metric, model):
    rows = P6_METRICS[(P6_METRICS["metric"] == metric) & (P6_METRICS["model"].str.startswith(model))]
    return rows["value"].iloc[0] if len(rows) else np.nan
display(P6_METRICS)
unrec = P6[P6["metric"].isna()]
if len(unrec):
    print(f"{len(unrec)} numeric values not recognised as metrics (set PHASE6_BASELINE in 1.2 if any of these are):")
    display(unrec[["source", "path", "value"]])
# Phase 6 latency (e.g. evaluated.e2e_latency_ms.{mean,50%,95%,99%,max} + latency_queries)
_PK = {"mean": "mean", "avg": "mean", "50%": "p50", "p50": "p50", "median": "p50", "95%": "p95", "p95": "p95",
       "99%": "p99", "p99": "p99", "max": "max"}
P6_LATENCY = {}
for r in P6.itertuples():
    parts = r.path.lower().split(".")
    if "latency" in r.path.lower():
        if parts[-1] in _PK: P6_LATENCY[_PK[parts[-1]]] = r.value
        elif "quer" in parts[-1] or parts[-1] in ("n", "count"): P6_LATENCY["n"] = int(r.value)
if P6_LATENCY: print("Phase 6 latency (ms):", P6_LATENCY)
if len(P6_METRICS):
    check("phase6_baseline_found", "PASS", f"{len(P6_METRICS)} retrieval metrics" + (" + latency" if P6_LATENCY else ""))
elif P6_LATENCY:
    check("phase6_baseline_found", "WARN", "Phase 6 recorded latency only (no retrieval scores) — compared in 8.2; "
          "retrieval baseline = Phase 9 pretrained run (set PHASE6_BASELINE if you have the Phase 6 scores)")
else:
    check("phase6_baseline_found", "FAIL", "no Phase 6 metrics found — set PHASE6_BASELINE in 1.2")

### 📘 3.2 Load the Phase 6 query set + labels

**Purpose:** Find and load the exact query set and labels Phase 6 used, so Phase 9 numbers are comparable.

**Expected Results:** Chosen file, number of queries, how many have id labels / attribute constraints / no-match labels. `eval_set_loaded` = PASS.

**Parameters:** `PHASE6_EVAL_PATH` (None = auto: paths in the `evaluated` block first, then `data/eval*`, `results/phase6/`, `src/rag/`).

In [ ]:
def eval_candidates():
    c = []
    if PHASE6_EVAL_PATH: c.append(REPO_LINK / PHASE6_EVAL_PATH)
    for _, s in string_leaves(EVALUATED):
        if re.search(r"\.(jsonl|json|csv|tsv)$", s):
            p = Path(s) if Path(s).is_absolute() else REPO_LINK / s
            c.append(p)
    pats = ["data/eval*/*.jsonl", "data/eval*/*.json", "data/eval*/*.csv", "data/*eval*.*", "data/*/*eval*.*",
            "data/*/*quer*.*", "eval*/*.jsonl", "eval*/*.json", "eval*/*.csv", "results/phase6/*.jsonl",
            "results/phase6/*.csv", "results/phase6/*quer*.json", "src/rag/*eval*.json*", "src/rag/*quer*.json*"]
    for pat in pats: c += sorted(REPO_LINK.glob(pat))
    return [p for p in dict.fromkeys(c) if p.exists() and p.is_file() and p.suffix in (".jsonl", ".json", ".csv", ".tsv")]

scored = []
for p in eval_candidates():
    try:
        es = M.normalize_eval_set(M.load_records(p))
    except Exception as e:
        continue
    n_lab = sum(bool(e["qrels"]) or bool(e["expected"]) for e in es)
    if es: scored.append((p, es, n_lab))
    print(f"  candidate {p.relative_to(REPO_LINK) if str(p).startswith(str(REPO_LINK)) else p}: {len(es)} queries, {n_lab} labelled")
assert scored, "No Phase 6 eval file found — set PHASE6_EVAL_PATH in 1.2"
explicit = [s for s in scored if PHASE6_EVAL_PATH and s[0] == REPO_LINK / PHASE6_EVAL_PATH]
EVAL_PATH, EVALSET, _ = explicit[0] if explicit else max(scored, key=lambda s: (s[2], len(s[1])))
summary = {"file": str(EVAL_PATH), "queries": len(EVALSET),
           "with_id_labels": sum(bool(e["qrels"]) for e in EVALSET),
           "with_attr_constraints": sum(bool(e["expected"]) for e in EVALSET),
           "with_no_match_label": sum(e["expect_no_match"] is not None for e in EVALSET),
           "multi_turn": sum(bool(e["history"]) for e in EVALSET)}
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
display(pd.DataFrame([{"qid": e["qid"], "query": e["query"], "n_rel": len(e["qrels"]), "expected": e["expected"]} for e in EVALSET[:8]]))
check("eval_set_loaded", "PASS" if summary["with_id_labels"] or summary["with_attr_constraints"] else "WARN",
      f"{summary['queries']} queries from {EVAL_PATH.name} ({summary['with_id_labels']} with id labels)")

### 📘 3.3 Mine the logs → evaluation cases

**Purpose:** Turn real usage into eval cases: thumbs-down and no-match requests from `logs/requests.jsonl` + `logs/feedback.jsonl`, after removing test sessions (`phase7-`, `phase8-`, `phase9-`, `test-`).

**Expected Results:** Log statistics (real vs filtered requests, 👍/👎, clicks, CTR, mean click rank, no-match rate) and the deduplicated case list saved to `results/phase9/log_cases.csv`. Thumbs-down cases are kept first.

**Parameters:** `TEST_SESSION_PREFIXES`, `MAX_LOG_CASES`, `INCLUDE_LOG_CASES`.

In [ ]:
MINED = L.mine(REPO_LINK / "logs" / "requests.jsonl", REPO_LINK / "logs" / "feedback.jsonl", TEST_SESSION_PREFIXES)
LOG_STATS = MINED["stats"]
display(pd.DataFrame([LOG_STATS]).T.rename(columns={0: "value"}))
LOG_CASES = MINED["cases"]
if not LOG_CASES.empty:
    LOG_CASES = (LOG_CASES.assign(_p=~LOG_CASES["reason"].str.contains("thumbs_down"))
                 .sort_values("_p", kind="stable").drop(columns="_p").head(MAX_LOG_CASES).reset_index(drop=True))
    LOG_CASES.to_csv(RESULTS_DIR / "log_cases.csv", index=False)
    display(LOG_CASES[["case_id", "reason", "query", "comment", "logged_no_match"]].head(20))
save_json(LOG_STATS, "log_stats.json")
if LOG_STATS["requests_total"] == 0:
    check("log_cases_mined", "WARN", "logs/requests.jsonl missing or empty")
else:
    check("log_cases_mined", "PASS" if len(LOG_CASES) else "WARN",
          ("" if len(LOG_CASES) else "no real-user signal yet — ") + f"{len(LOG_CASES)} cases from {LOG_STATS['requests_real']} real requests "
          f"({LOG_STATS['requests_test_filtered']} test requests filtered)")

# combined case list used by sections 6–10
CASES = [{"case_id": e["qid"], "source": "phase6", **{k: e[k] for k in ("query", "qrels", "expected", "expect_no_match", "history")}}
         for e in EVALSET]
if INCLUDE_LOG_CASES and not LOG_CASES.empty:
    CASES += [{"case_id": r.case_id, "source": f"log:{r.reason}", "query": r.query, "qrels": {}, "expected": {},
               "expect_no_match": None, "history": None} for r in LOG_CASES.itertuples()]
print(f"{len(CASES)} cases in total ({sum(c['source'] == 'phase6' for c in CASES)} Phase 6 + "
      f"{sum(c['source'] != 'phase6' for c in CASES)} from logs)")

## 4. Retrieval precision: pretrained vs fine-tuned E5 (plan 9.1)

### 📘 4.1 Resolve the embedding setup and load the Chroma index

**Purpose:** Find the base E5 model and the LoRA adapter, open the local Chroma copy and pull every product's id, document text, metadata and stored embedding.

**Expected Results:** Base model name, adapter dir, collection name and `18,192` products. `chroma_count` = PASS when the count matches `EXPECTED_N_PRODUCTS`.

**Parameters:** `EMBED_BASE`, `EMBED_ADAPTER`, `CHROMA_DIR`, `CHROMA_COLLECTION`, `RUN_RETRIEVAL_COMPARISON`.

In [ ]:
import chromadb

def find_adapter():
    if EMBED_ADAPTER: return LOCAL_RUN / EMBED_ADAPTER if not Path(EMBED_ADAPTER).is_absolute() else Path(EMBED_ADAPTER)
    for path, s in string_leaves(local_cfg):
        key = "/".join(map(str, path)).lower() + " " + s.lower()
        if any(t in key for t in ("lora", "adapter", "finetun", "fine_tun", "fine-tun")):
            p = Path(s) if Path(s).is_absolute() else LOCAL_RUN / s
            if p.exists(): return p
    hits = sorted(LOCAL_RUN.glob("**/adapter_config.json"))
    return hits[0].parent if hits else None

EMB_ADAPTER_DIR = find_adapter()
ADAPTER_CFG = None
if EMB_ADAPTER_DIR:
    ac = next(iter(sorted(Path(EMB_ADAPTER_DIR).rglob("adapter_config.json"))), None)
    if ac: ADAPTER_CFG = json.loads(ac.read_text()); EMB_ADAPTER_DIR = ac.parent
EMB_BASE = EMBED_BASE or (ADAPTER_CFG or {}).get("base_model_name_or_path") or next(
    (s for _, s in string_leaves(RAG_CFG) if "e5" in s.lower() and "/" in s and not (DRIVE_PROJECT / s).exists()), "intfloat/e5-small-v2")
print("base model :", EMB_BASE)
print("adapter dir:", EMB_ADAPTER_DIR, "(LoRA)" if ADAPTER_CFG else "(full model)" if EMB_ADAPTER_DIR else "NOT FOUND")

assert LOCAL_CHROMA_DIRS, "No local Chroma copy — check 1.5 / CHROMA_DIR"
chroma_client = chromadb.PersistentClient(path=str(LOCAL_CHROMA_DIRS[0]))
names = [c if isinstance(c, str) else c.name for c in chroma_client.list_collections()]
coll_name = CHROMA_COLLECTION or next((s for p, s in string_leaves(RAG_CFG) if "collection" in str(p[-1]).lower() and s in names), None) or names[0]
COLL = chroma_client.get_collection(coll_name)
N_DOCS = COLL.count()
print(f"collection '{coll_name}' (of {names}) → {N_DOCS:,} items")

DOC_IDS, DOC_TEXT, DOC_META, DOC_EMB = [], [], [], []
for off in range(0, N_DOCS, 2000):
    r = COLL.get(limit=2000, offset=off, include=["documents", "metadatas", "embeddings"])
    DOC_IDS += r["ids"]; DOC_TEXT += r["documents"] or [None] * len(r["ids"])
    DOC_META += r["metadatas"] or [{}] * len(r["ids"]); DOC_EMB.append(np.asarray(r["embeddings"], dtype=np.float32))
DOC_EMB = np.vstack(DOC_EMB); DOC_EMB /= np.linalg.norm(DOC_EMB, axis=1, keepdims=True) + 1e-12
ID2IDX = {d: i for i, d in enumerate(DOC_IDS)}
print("stored embeddings:", DOC_EMB.shape, "| example doc:", (DOC_TEXT[0] or "")[:160])
check("chroma_count", "PASS" if N_DOCS == EXPECTED_N_PRODUCTS else "WARN", f"{N_DOCS:,} items (expected {EXPECTED_N_PRODUCTS:,})")
lab_ids = {pid for e in EVALSET for pid in e["qrels"]}
if lab_ids:
    cov = sum(p in ID2IDX for p in lab_ids) / len(lab_ids)
    print(f"label ids present in the index: {cov:.1%}")

### 📘 4.2 Embed with both models (and verify the index)

**Purpose:** Load the pretrained and the fine-tuned E5. First re-embed 256 random products with the fine-tuned model and compare with the stored Chroma vectors — this verifies the adapter loading and auto-detects the passage prefix. Then embed all products with the pretrained model and all queries with both.

**Expected Results:** `index_reembed_cosine` ≥ 0.99 (PASS) → the fine-tuned side reuses the stored index vectors exactly. ≈ 1–2 min on a T4.

**Parameters:** `QUERY_PREFIX`, `PASSAGE_PREFIX`, `EMBED_BATCH`.

In [ ]:
from sentence_transformers import SentenceTransformer
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_finetuned():
    if EMB_ADAPTER_DIR is None: raise FileNotFoundError("fine-tuned adapter not found — set EMBED_ADAPTER")
    if ADAPTER_CFG is not None:
        m = SentenceTransformer(EMB_BASE, device=DEVICE)
        try:
            m.load_adapter(str(EMB_ADAPTER_DIR))
        except Exception as e:
            print("load_adapter failed (", e, ") → PeftModel merge")
            from peft import PeftModel
            m[0].auto_model = PeftModel.from_pretrained(m[0].auto_model, str(EMB_ADAPTER_DIR)).merge_and_unload()
        return m
    return SentenceTransformer(str(EMB_ADAPTER_DIR), device=DEVICE)

def embed(model, texts, prefix=""):
    texts = [t if t.startswith(prefix) else prefix + t for t in texts] if prefix else list(texts)
    return model.encode(texts, batch_size=EMBED_BATCH, normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=len(texts) > 1000).astype(np.float32)

if RUN_RETRIEVAL_COMPARISON:
    ft_model = load_finetuned()
    rng = np.random.default_rng(0)
    sample = [i for i in rng.choice(len(DOC_IDS), size=min(256, len(DOC_IDS)), replace=False) if DOC_TEXT[i]]
    cands = [PASSAGE_PREFIX] if PASSAGE_PREFIX is not None else ["passage: ", ""]
    cos = {p: float((embed(ft_model, [DOC_TEXT[i] for i in sample], p) * DOC_EMB[sample]).sum(1).mean()) for p in cands}
    P_PREFIX = max(cos, key=cos.get)
    print("re-embed cosine vs stored index:", {repr(k): round(v, 4) for k, v in cos.items()}, "→ passage prefix", repr(P_PREFIX))
    check("index_reembed_cosine", "PASS" if cos[P_PREFIX] >= 0.99 else "WARN",
          f"{cos[P_PREFIX]:.4f} (fine-tuned model vs stored Chroma vectors)")
    if cos[P_PREFIX] >= 0.99:
        FT_DOC = DOC_EMB
    else:
        print("⚠️ stored vectors differ → re-embedding the full catalog with the fine-tuned model for a fair comparison")
        FT_DOC = embed(ft_model, [t or "" for t in DOC_TEXT], P_PREFIX)
    pre_model = SentenceTransformer(EMB_BASE, device=DEVICE)
    PRE_DOC = embed(pre_model, [t or "" for t in DOC_TEXT], P_PREFIX)
    labelled = [e for e in EVALSET if e["qrels"] or e["expected"]]
    qtexts = [e["query"] for e in labelled]
    FT_Q, PRE_Q = embed(ft_model, qtexts, QUERY_PREFIX), embed(pre_model, qtexts, QUERY_PREFIX)
    print("query embeddings:", FT_Q.shape)
else:
    check("index_reembed_cosine", "SKIP", "RUN_RETRIEVAL_COMPARISON=False")

### 📘 4.3 Score pretrained vs fine-tuned and compare with Phase 6

**Purpose:** Dense top-k retrieval (cosine) for every labelled Phase 6 query with both models, scored with the Phase 6 metrics; paired bootstrap CI for the fine-tuned − pretrained difference; reproduction check against the Phase 6 `evaluated` numbers; sanity check that our numpy ranking equals Chroma's own query results.

**Expected Results:** Table metric × {pretrained, finetuned, Δ, Phase 6}; `retrieval_ft_beats_pretrained` = PASS when the CI of Δ precision@5 is above 0; `phase6_reproduced` = PASS when |Phase 9 − Phase 6| ≤ `REPRO_TOLERANCE`.

**Parameters:** `KS`, `PRIMARY_K`, `RELEVANCE_THRESHOLD`, `REPRO_TOLERANCE`.

In [ ]:
def topk(qe, de, k):
    sims = qe @ de.T
    idx = np.argpartition(-sims, kth=min(k, sims.shape[1] - 1), axis=1)[:, :k]
    order = np.take_along_axis(sims, idx, 1).argsort(1)[:, ::-1]
    return np.take_along_axis(idx, order, 1)

if RUN_RETRIEVAL_COMPARISON:
    K = max(KS)
    runs, metas = {}, {}
    for tag, qe, de in (("pretrained", PRE_Q, PRE_DOC), ("finetuned", FT_Q, FT_DOC)):
        top = topk(qe, de, K)
        runs[tag] = {e["qid"]: [DOC_IDS[j] for j in row] for e, row in zip(labelled, top)}
        metas[tag] = {e["qid"]: [DOC_META[j] or {} for j in row] for e, row in zip(labelled, top)}
    PERQ = {t: M.score_runs(runs[t], labelled, KS, RELEVANCE_THRESHOLD, metas[t]) for t in runs}
    RET_MEAN = {t: M.mean_metrics(PERQ[t]) for t in PERQ}
    RET_CMP = pd.DataFrame(RET_MEAN).rename_axis("metric").reset_index()
    RET_CMP["delta"] = RET_CMP["finetuned"] - RET_CMP["pretrained"]
    display(RET_CMP.round(4))

    primary = f"precision@{PRIMARY_K}" if f"precision@{PRIMARY_K}" in PERQ["finetuned"] else f"attr_precision@{PRIMARY_K}"
    BOOT = {m: M.paired_bootstrap(PERQ["pretrained"][m], PERQ["finetuned"][m])
            for m in (primary, f"ndcg@{PRIMARY_K}", "mrr") if m in PERQ["finetuned"]}
    display(pd.DataFrame(BOOT).T.round(4))
    b = BOOT[primary]
    others = "; ".join(f"Δ{m} {v['delta']:+.3f} [{v['ci_low']:+.3f}, {v['ci_high']:+.3f}]" for m, v in BOOT.items() if m != primary)
    check("retrieval_ft_beats_pretrained", "PASS" if b["ci_low"] > 0 else "WARN" if b["delta"] >= 0 else "FAIL",
          f"Δ{primary} = {b['delta']:+.3f} (95% CI {b['ci_low']:+.3f}…{b['ci_high']:+.3f}, n={b['n']}); {others}")

    # numpy ranking vs Chroma's own query (fine-tuned): validates distance metric + index
    n = min(20, len(labelled))
    res = COLL.query(query_embeddings=FT_Q[:n].tolist(), n_results=PRIMARY_K, include=[])
    ov = np.mean([len(set(res["ids"][i]) & set(runs["finetuned"][labelled[i]["qid"]][:PRIMARY_K])) / PRIMARY_K for i in range(n)])
    check("chroma_consistency", "PASS" if ov >= 0.8 else "WARN", f"top-{PRIMARY_K} overlap numpy vs Chroma = {ov:.2f}")

    pd.concat([PERQ[t].assign(model=t) for t in PERQ]).to_csv(RESULTS_DIR / "retrieval_per_query.csv", index=False)
    save_json(BOOT, "retrieval_bootstrap.json")
    del ft_model, pre_model, PRE_DOC
    if torch.cuda.is_available(): torch.cuda.empty_cache()

### 📘 4.4 Compare with the Phase 6 numbers

**Purpose:** Put the Phase 6 `evaluated` values next to the Phase 9 in-process results. Kept separate from 4.3 so it can be re-run alone after fixing `PHASE6_BASELINE` (4.3 frees the models when it finishes).

**Expected Results:** `RET_CMP` with `phase6_pretrained` / `phase6_finetuned` columns; `phase6_reproduced` = PASS when |Phase 9 − Phase 6| ≤ `REPRO_TOLERANCE` on the primary metric.

**Parameters:** `REPRO_TOLERANCE`, `PHASE6_BASELINE`.

In [ ]:
if RUN_RETRIEVAL_COMPARISON:
    RET_CMP["phase6_pretrained"] = RET_CMP["metric"].map(lambda m: p6_value(m, "pretrained"))
    RET_CMP["phase6_finetuned"] = RET_CMP["metric"].map(lambda m: p6_value(m, "finetuned"))
    shown = RET_CMP[~RET_CMP["metric"].str.startswith("judged")]
    display(shown.round(4))
    RET_CMP.to_csv(RESULTS_DIR / "retrieval_pretrained_vs_finetuned.csv", index=False)
    rc = RET_CMP.set_index("metric")
    msgs, worst = [], 0.0
    for model in ("pretrained", "finetuned"):
        p6v, p9v = rc.loc[primary, f"phase6_{model}"], rc.loc[primary, model]
        if not np.isnan(p6v):
            msgs.append(f"{model} {p9v:.3f} vs {p6v:.3f}"); worst = max(worst, abs(p9v - p6v))
    if not msgs:
        check("phase6_reproduced", "SKIP" if P6_LATENCY else "FAIL", f"no Phase 6 value for {primary} — set PHASE6_BASELINE in 1.2; Phase 9 fine-tuned = {rc.loc[primary, 'finetuned']:.3f}")
    else:
        check("phase6_reproduced", "PASS" if worst <= REPRO_TOLERANCE else "WARN",
              f"{primary} Phase 9 vs Phase 6: " + ", ".join(msgs) + " (dense-only, no metadata filters)")

## 5. Start the Phase 7 API

### 📘 5.1 Start the API subprocess and wait for `/health`

**Purpose:** Run uvicorn from the local copy (GPU model loads once), wait until `/health` reports `ok`.

**Expected Results:** `✅ /health ok after …s` (≈ 1–3 min for Qwen2.5-3B). On failure the last uvicorn log lines are printed.

**Parameters:** `API_APP`, `API_PORT`, `USE_FAKE`, `API_ENV`, `HEALTH_TIMEOUT_S`.

In [ ]:
from src.eval.api_server import ApiServer, detect_app_module
from src.eval.client import ShopTalkClient
app_mod, is_factory = (API_APP, False) if API_APP else detect_app_module(LOCAL_RUN)
if USE_FAKE: os.environ["SHOPTALK_FAKE"] = "1"
else: os.environ.pop("SHOPTALK_FAKE", None)
API_ENV_ALL = dict(API_ENV, **({"HF_HOME": HF_HOME} if HF_HOME else {}))
print("app:", app_mod, "| fake:", USE_FAKE)

def start_api():
    srv = ApiServer(LOCAL_RUN, app_mod, API_PORT, API_ENV_ALL, LOCAL_RUN / "logs" / "phase9_uvicorn.log", is_factory)
    srv.start()
    cl = ShopTalkClient(srv.url, query_field=QUERY_FIELD)
    try:
        h = cl.wait_healthy(HEALTH_TIMEOUT_S, proc=srv.proc)
    except Exception:
        print(srv.tail(60)); raise
    return srv, cl, h

SERVER, CLIENT, HEALTH = start_api()
check("api_health_ok", "PASS", HEALTH)

### 📘 5.2 Smoke request + response schema

**Purpose:** One `/recommend` call; verify the Phase 7 response contract.

**Expected Results:** The response keys; `api_schema` = PASS when `request_id, answer, products, no_match, latency_ms` are present.

**Parameters:** none

In [ ]:
LAT_KEYS = ("parse", "retrieve", "generate", "queue", "total")
def flat(resp):
    lat = resp.get("latency_ms") or {}
    prods = resp.get("products") or []
    return {"request_id": resp.get("request_id"), "answer": resp.get("answer") or "", "products": prods,
            "product_ids": [str(p.get("id")) for p in prods], "no_match": bool(resp.get("no_match")),
            "unmet_constraints": resp.get("unmet_constraints") or [], "client_ms": resp.get("_client_ms"),
            "http_status": resp.get("_http_status"), "error": resp.get("error") or resp.get("detail"),
            **{f"lat_{k}": lat.get(k) for k in LAT_KEYS}}

smoke = CLIENT.recommend(CASES[0]["query"], f"{SESSION_PREFIX}-smoke")
print("keys:", sorted(smoke))
print("answer:", (smoke.get("answer") or "")[:300])
missing = [k for k in ("request_id", "answer", "products", "no_match", "latency_ms") if k not in smoke]
check("api_schema", "PASS" if not missing and smoke["_http_status"] == 200 else "FAIL",
      f"http {smoke['_http_status']}, missing={missing}, client {smoke['_client_ms']:.0f} ms")

## 6. End-to-end evaluation through the API

### 📘 6.1 Run every case through `/recommend`

**Purpose:** Send each Phase 6 query and each log case to the API (one session per case, so no history leaks between cases). Raw responses are saved for scoring, rating and the failure write-up.

**Expected Results:** `results/phase9/api_eval_runs.jsonl`; `api_eval_errors` = PASS when every request returns HTTP 200. ≈ 4–5 s per case.

**Parameters:** none

In [ ]:
from tqdm.auto import tqdm
API_RUNS = []
for c in tqdm(CASES, desc="eval"):
    r = flat(CLIENT.recommend(c["query"], f"{SESSION_PREFIX}-eval-{c['case_id']}", history=c.get("history")))
    API_RUNS.append({"case_id": c["case_id"], "source": c["source"], "query": c["query"], **r})
with open(RESULTS_DIR / "api_eval_runs.jsonl", "w") as f:
    for r in API_RUNS: f.write(json.dumps(r, default=str) + "\n")
errs = [r for r in API_RUNS if r["http_status"] != 200]
check("api_eval_errors", "PASS" if not errs else "FAIL", f"{len(errs)} errors / {len(API_RUNS)} requests")

### 📘 6.2 Retrieval through the API vs Phase 6

**Purpose:** Score the products the full pipeline returns (parser + filters + fine-tuned retrieval) with the same metrics, and the no-match decision where the eval set labels it.

**Expected Results:** Table: metric · API (end-to-end) · in-process fine-tuned (§4) · Phase 6. `api_retrieval_vs_phase6` = PASS when the API's primary metric is within `REPRO_TOLERANCE` of, or above, Phase 6.

**Parameters:** `KS`, `PRIMARY_K`.

In [ ]:
RUN_BY_ID = {r["case_id"]: r for r in API_RUNS}
p6_cases = [e for e in EVALSET if e["qid"] in RUN_BY_ID]
api_runs = {e["qid"]: RUN_BY_ID[e["qid"]]["product_ids"] for e in p6_cases}
api_metas = {e["qid"]: RUN_BY_ID[e["qid"]]["products"] for e in p6_cases}
API_PERQ = M.score_runs(api_runs, [e for e in p6_cases if e["qrels"] or e["expected"]], KS, RELEVANCE_THRESHOLD, api_metas)
API_MEAN = M.mean_metrics(API_PERQ) if len(API_PERQ) else {}
cmp = pd.DataFrame({"api_end_to_end": API_MEAN})
if RUN_RETRIEVAL_COMPARISON:
    cmp["inprocess_finetuned"] = pd.Series(RET_MEAN["finetuned"])
    cmp["inprocess_pretrained"] = pd.Series(RET_MEAN["pretrained"])
cmp["phase6"] = [p6_value(m, "finetuned") for m in cmp.index]
API_CMP = cmp.rename_axis("metric").reset_index()
display(API_CMP.round(4))
API_CMP.to_csv(RESULTS_DIR / "api_retrieval_vs_phase6.csv", index=False); API_PERQ.to_csv(RESULTS_DIR / "api_retrieval_per_query.csv", index=False)

nm = [(e["expect_no_match"], RUN_BY_ID[e["qid"]]["no_match"]) for e in p6_cases if e["expect_no_match"] is not None]
NO_MATCH_ACC = float(np.mean([a == b for a, b in nm])) if nm else np.nan
print(f"no-match accuracy: {NO_MATCH_ACC:.3f} on {len(nm)} labelled queries" if nm else "no no-match labels in the eval set")

primary_api = f"precision@{PRIMARY_K}" if f"precision@{PRIMARY_K}" in API_MEAN else f"attr_precision@{PRIMARY_K}"
a, p = API_MEAN.get(primary_api, np.nan), API_CMP.set_index("metric")["phase6"].get(primary_api, np.nan)
if np.isnan(p):
    check("api_retrieval_vs_phase6", "SKIP" if P6_LATENCY else "WARN",
          f"API {primary_api} = {a:.3f}; Phase 6 recorded no retrieval scores (compare with in-process fine-tuned in the table)")
else:
    check("api_retrieval_vs_phase6", "PASS" if a >= p - REPRO_TOLERANCE else "WARN",
          f"API {primary_api} = {a:.3f} vs Phase 6 {p:.3f}")

### 📘 6.3 Automatic answer checks

**Purpose:** Cheap, objective checks on every answer: are the cited `[ID: …]` among the returned products, does the answer cite anything, are quoted prices in the product cards, does it end mid-sentence?

**Expected Results:** Summary table; `no_hallucinated_ids` = PASS at 0%, WARN ≤ 2%, FAIL above; `citations_present` = PASS when ≥ 90% of answers with products cite at least one.

**Parameters:** none

In [ ]:
PROXY = pd.DataFrame([{"case_id": r["case_id"], "source": r["source"],
                       **Q.answer_proxies(r["answer"], r["products"], r["no_match"])} for r in API_RUNS])
PROXY_SUM = Q.summarize(PROXY)
display(pd.DataFrame([PROXY_SUM]).T.rename(columns={0: "value"}).round(3))
PROXY.to_csv(RESULTS_DIR / "answer_proxies.csv", index=False); save_json(PROXY_SUM, "answer_proxies_summary.json")
h = PROXY_SUM["hallucinated_id_rate"]
check("no_hallucinated_ids", "PASS" if h == 0 else "WARN" if h <= 0.02 else "FAIL", f"{h:.1%} of answers cite an id not returned")
c = PROXY_SUM["cites_any_rate"]
check("citations_present", "PASS" if c >= 0.9 else "WARN", f"{c:.1%} of answers with products cite ≥ 1 [ID: …]")

### 📘 6.4 Re-run of the log cases

**Purpose:** Did today's pipeline fix what users complained about? Compare each log case's original outcome (logged products / no-match) with the new response.

**Expected Results:** Per-case table (old vs new no-match, product overlap) and counts; these cases are over-sampled in the manual rating (§7) because they have no labels.

**Parameters:** none

In [ ]:
if not LOG_CASES.empty:
    rows = []
    for lc in LOG_CASES.itertuples():
        new = RUN_BY_ID.get(lc.case_id, {})
        old_ids, new_ids = set(map(str, lc.logged_product_ids or [])), set(new.get("product_ids", []))
        rows.append({"case_id": lc.case_id, "reason": lc.reason, "query": lc.query, "comment": lc.comment,
                     "old_no_match": lc.logged_no_match, "new_no_match": new.get("no_match"),
                     "product_overlap": len(old_ids & new_ids) / max(len(old_ids | new_ids), 1)})
    LOG_RERUN = pd.DataFrame(rows)
    display(LOG_RERUN)
    nm = LOG_RERUN[LOG_RERUN["old_no_match"]]
    print(f"no-match cases now answered: {(~nm['new_no_match'].astype(bool)).sum()}/{len(nm)}")
    LOG_RERUN.to_csv(RESULTS_DIR / "log_cases_rerun.csv", index=False)
else:
    LOG_RERUN = pd.DataFrame(); print("no log cases")

## 7. Generation quality — manual rating (plan 9.2)

### 📘 7.1 Build the rating sheet

**Purpose:** Sample `N_RATE` answers (log cases first — thumbs-down, then no-match — up to a quarter of the sheet, the rest random Phase 6 queries) into `results/phase9/rating_sheet.csv` with the product cards next to each answer. If the sheet already exists it is kept, so ratings survive re-runs.

**Expected Results:** `rating_sheet.csv` with `N_RATE` rows; columns relevance / groundedness / fluency / notes empty.

**Parameters:** `N_RATE`, `RATING_SEED`.

In [ ]:
RATING_SHEET = RESULTS_DIR / "rating_sheet.csv"
def products_text(prods, n=5):
    out = []
    for p in prods[:n]:
        bits = [str(p.get(k)) for k in ("brand", "price", "color", "product_type") if p.get(k) not in (None, "")]
        out.append(f"[{p.get('id')}] {p.get('title', '')} — {' · '.join(bits)}")
    return "\n".join(out) or "(no products)"

if RATING_SHEET.exists():
    sheet = pd.read_csv(RATING_SHEET)
    print(f"Keeping existing sheet: {len(sheet)} rows, {sheet[['relevance','groundedness','fluency']].notna().all(1).sum()} rated")
else:
    rnd = random.Random(RATING_SEED)
    logs = sorted([r for r in API_RUNS if r["source"].startswith("log")], key=lambda r: "thumbs_down" not in r["source"])
    pick = logs[: N_RATE // 4]
    p6r = [r for r in API_RUNS if r["source"] == "phase6"]; rnd.shuffle(p6r)
    pick += p6r[: N_RATE - len(pick)]
    rnd.shuffle(pick)
    sheet = pd.DataFrame([{"case_id": r["case_id"], "source": r["source"], "query": r["query"],
                           "products_text": products_text(r["products"]), "answer": r["answer"],
                           "relevance": None, "groundedness": None, "fluency": None, "notes": ""} for r in pick])
    sheet.to_csv(RATING_SHEET, index=False)
    print(f"Wrote {len(sheet)} rows → {RATING_SHEET}")
display(sheet[["case_id", "source", "query"]].head(10))

### 📘 7.2 Rate the answers (interactive)

**Purpose:** Rate each answer 1–5 on **relevance** (do the products and the answer fit every stated need?), **groundedness** (is every claim — price, colour, brand, features, IDs — in the product cards?) and **fluency** (clear, natural, well organised). Every click saves to Drive; re-run the cell to resume. Alternatively download `rating_sheet.csv`, fill it in a spreadsheet and upload it back to the same path.

**Expected Results:** A widget showing query, product cards and answer; the progress line reaches `N_RATE/N_RATE rated`.

**Parameters:** rubric in `src/eval/rater.py`.

In [ ]:
_ = R.make_rater(RATING_SHEET)

### 📘 7.3 Summarise the ratings

**Purpose:** Mean, share ≥ 4 and distribution per dimension, overall and by source (Phase 6 vs log cases).

**Expected Results:** `manual_ratings_count` = PASS with ≥ `MIN_RATINGS` fully rated rows (else PENDING); `manual_quality` = PASS when every mean meets `QUALITY_TARGETS`.

**Parameters:** `MIN_RATINGS`, `QUALITY_TARGETS`.

In [ ]:
DIMS = ["relevance", "groundedness", "fluency"]
RATINGS = pd.read_csv(RATING_SHEET)
rated = RATINGS.dropna(subset=DIMS).copy()
for d in DIMS: rated[d] = rated[d].astype(float)
if len(rated):
    RATING_SUM = pd.DataFrame({d: {"mean": rated[d].mean(), "share_ge4": (rated[d] >= 4).mean(), "min": rated[d].min(),
                                   **{f"n_{k}": int((rated[d] == k).sum()) for k in range(1, 6)}} for d in DIMS}).T
    display(RATING_SUM.round(3))
    display(rated.assign(group=rated["source"].str.split(":").str[0]).groupby("group")[DIMS].agg(["mean", "count"]).round(2))
    save_json({"n_rated": len(rated), **RATING_SUM["mean"].to_dict()}, "manual_rating_summary.json")
else:
    RATING_SUM = pd.DataFrame()
check("manual_ratings_count", "PASS" if len(rated) >= MIN_RATINGS else "PENDING", f"{len(rated)}/{len(RATINGS)} rated (need {MIN_RATINGS})")
if len(rated) >= MIN_RATINGS:
    miss = {d: round(float(rated[d].mean()), 2) for d in DIMS if rated[d].mean() < QUALITY_TARGETS[d]}
    check("manual_quality", "PASS" if not miss else "WARN",
          "means " + ", ".join(f"{d}={rated[d].mean():.2f}" for d in DIMS) + (f" — below target: {miss}" if miss else ""))
else:
    check("manual_quality", "PENDING", "rate the sheet in 7.2, then re-run this cell")

## 8. Latency (plan 9.3)

### 📘 8.1 120 sequential `/recommend` requests

**Purpose:** Measure end-to-end latency with the server-side stage breakdown. Requests are sequential (one user at a time, like Phase 7), warm-up requests are excluded, queries cycle through the shuffled Phase 6 set.

**Expected Results:** `results/phase9/latency_raw.csv` with ≥ 100 rows; ≈ 8–10 min.

**Parameters:** `N_LATENCY`, `LATENCY_WARMUP`.

In [ ]:
lat_queries = [c["query"] for c in CASES if c["source"] == "phase6"]
random.Random(1).shuffle(lat_queries)
def lat_call(q, i):
    r = flat(CLIENT.recommend(q, f"{SESSION_PREFIX}-lat-{i}"))
    return {k: v for k, v in r.items() if k not in ("products", "answer")} | {"answer_words": len(r["answer"].split())}
LAT_RAW = pd.DataFrame(LT.run_requests(lat_call, lat_queries, N_LATENCY, LATENCY_WARMUP, desc="latency"))
LAT_RAW["api_overhead"] = LAT_RAW["client_ms"] - pd.to_numeric(LAT_RAW["lat_total"], errors="coerce")
LAT_RAW.to_csv(RESULTS_DIR / "latency_raw.csv", index=False)
ok = LAT_RAW[LAT_RAW["ok"] & (LAT_RAW["http_status"] == 200)]
check("latency_n", "PASS" if len(ok) >= 100 else "FAIL", f"{len(ok)} successful requests")

### 📘 8.2 P50 / P95 / P99 per stage

**Purpose:** Report retrieval and generation separately (plan 9.3), plus parse, queue, server total, client round-trip and the API overhead; compare with Phase 7.

**Expected Results:** Percentile table; generation should dominate (~93% in Phase 7). `latency_p95_client` = PASS when client P95 < `LATENCY_TARGET_MS`.

**Parameters:** `LATENCY_TARGET_MS`, `PHASE7_CLIENT_P95_MS`.

In [ ]:
LAT_TABLE = LT.latency_table(ok, {"parse": "lat_parse", "retrieve": "lat_retrieve", "generate": "lat_generate",
                                  "queue": "lat_queue", "server total": "lat_total", "client round-trip": "client_ms",
                                  "API overhead": "api_overhead"})
display(LAT_TABLE)
LAT_TABLE.to_csv(RESULTS_DIR / "latency_summary.csv", index=False)
import torch
gen_share = (pd.to_numeric(ok["lat_generate"], errors="coerce") / pd.to_numeric(ok["lat_total"], errors="coerce")).median()
p95 = LAT_TABLE.set_index("stage").loc["client round-trip", "p95"]
print(f"generation share of server time (median): {gen_share:.1%} | client P95 {p95:,.0f} ms vs Phase 7 {PHASE7_CLIENT_P95_MS:,} ms")
try:
    GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
except Exception:
    GPU_NAME = "unknown"
check("latency_p95_client", "PASS" if p95 < LATENCY_TARGET_MS else "FAIL", f"P95 {p95:,.0f} ms (target < {LATENCY_TARGET_MS:,}) on {GPU_NAME}")
if P6_LATENCY:
    st = LAT_TABLE.set_index("stage")
    LAT_VS_P6 = pd.DataFrame({"phase6_e2e": {k: P6_LATENCY.get(k) for k in ("mean", "p50", "p95", "p99", "max")},
                              "phase9_server_total": {k: st.loc["server total", k] for k in ("mean", "p50", "p95", "p99", "max")},
                              "phase9_client": {k: st.loc["client round-trip", k] for k in ("mean", "p50", "p95", "p99", "max")}})
    LAT_VS_P6["Δ client vs P6"] = LAT_VS_P6["phase9_client"] - LAT_VS_P6["phase6_e2e"]
    print(f"Phase 6 (n={P6_LATENCY.get('n', '?')}) vs Phase 9 on {GPU_NAME} — compare like-for-like only if both ran on the same GPU:")
    display(LAT_VS_P6.round(0))
    LAT_VS_P6.to_csv(RESULTS_DIR / "latency_vs_phase6.csv")
    d = LAT_VS_P6.loc["p95", "Δ client vs P6"]
    check("latency_vs_phase6", "PASS" if d <= 0 else "WARN",
          f"client P95 {LAT_VS_P6.loc['p95', 'phase9_client']:,.0f} ms on {GPU_NAME} vs Phase 6 e2e P95 {P6_LATENCY.get('p95', float('nan')):,.0f} ms ({d:+,.0f} ms)")
check("latency_stage_breakdown", "PASS" if {"retrieve", "generate"} <= set(LAT_TABLE["stage"]) else "WARN",
      "retrieve + generate percentiles reported" if {"retrieve", "generate"} <= set(LAT_TABLE["stage"]) else "latency_ms lacks retrieve/generate")

### 📘 8.3 Streaming: first token and full answer (Phase 8 examples)

**Purpose:** Reproduce the Phase 8 warning: stream the 5 UI example queries and measure time-to-first-token and full-answer time. This is the baseline for the `max_new_tokens` sweep.

**Expected Results:** TTFT P50 ≈ 0.8 s and full-answer P95 ≈ 5.1–5.5 s if nothing changed since Phase 8. `stream_full_p95_baseline` = PASS/WARN against the 5 s target.

**Parameters:** `PHASE8_EXAMPLES`, `N_STREAM`.

In [ ]:
def find_ui_examples(ui_dir):
    for f in sorted(Path(ui_dir).glob("*.py")):
        try: tree = ast.parse(f.read_text())
        except Exception: continue
        for node in ast.walk(tree):
            val = None
            if isinstance(node, (ast.Assign, ast.AnnAssign)):
                tg = node.targets if isinstance(node, ast.Assign) else [node.target]
                if any(isinstance(t, ast.Name) and "EXAMPLE" in t.id.upper() for t in tg): val = node.value
            elif isinstance(node, ast.keyword) and node.arg == "examples":
                val = node.value
            if val is None: continue
            try: lit = ast.literal_eval(val)
            except Exception: continue
            if isinstance(lit, (list, tuple)):
                qs = [x[0] if isinstance(x, (list, tuple)) and x else x for x in lit]
                qs = [q for q in qs if isinstance(q, str) and q.strip()]
                if qs: return qs, f.name
    return [], None

EXAMPLES, src_file = (PHASE8_EXAMPLES, "config") if PHASE8_EXAMPLES else find_ui_examples(LOCAL_RUN / "src" / "ui")
if not EXAMPLES:
    EXAMPLES, src_file = [c["query"] for c in CASES[:5]], "first 5 eval queries (fallback)"
print(f"{len(EXAMPLES)} example queries from {src_file}:", *EXAMPLES, sep="\n  ")

def stream_call(q, i):
    s = CLIENT.recommend_stream(q, f"{SESSION_PREFIX}-stream-{i}")
    return {"ttft_ms": s["ttft_ms"], "full_ms": s["total_ms"], "answer_words": len((s["answer"] or "").split()),
            "error": s["error"]}
STREAM_RAW = pd.DataFrame(LT.run_requests(stream_call, EXAMPLES, N_STREAM, warmup=2, desc="stream"))
STREAM_RAW.to_csv(RESULTS_DIR / "stream_raw.csv", index=False)
STREAM_TABLE = LT.latency_table(STREAM_RAW, {"first token": "ttft_ms", "full answer": "full_ms"})
display(STREAM_TABLE)
display(STREAM_RAW.groupby("query")[["ttft_ms", "full_ms", "answer_words"]].median().round(0))
fp95 = STREAM_TABLE.set_index("stage").loc["full answer", "p95"]
check("stream_full_p95_baseline", "PASS" if fp95 < LATENCY_TARGET_MS else "WARN", f"full-answer P95 {fp95:,.0f} ms at the current max_new_tokens")

### 📘 8.4 `/metrics` snapshot

**Purpose:** Save the API's own counters for the record.

**Expected Results:** `results/phase9/api_metrics.txt`.

**Parameters:** none

In [ ]:
try:
    txt = CLIENT.metrics(); (RESULTS_DIR / "api_metrics.txt").write_text(txt); print(txt[:1500])
except Exception as e:
    print("metrics unavailable:", e)

## 9. `max_new_tokens` sweep (Phase 8 follow-up)

### 📘 9.1 Run the sweep

**Purpose:** For each `max_new_tokens` value: write it into the **local** rag_config copy, restart the API, stream the Phase 8 examples (× `SWEEP_REPEATS`) plus a few eval queries, and record latency and answer checks. The original local config is restored afterwards; the Drive config is never touched here.

**Expected Results:** `results/phase9/sweep_raw.csv`. ≈ 3–5 min per value (model reload + ~40 streams). `sweep_override_effective` = PASS when answers actually get shorter at lower values.

**Parameters:** `RUN_SWEEP`, `SWEEP_VALUES` (None = current × 1.28 / 0.85 / 0.7 / 0.55, rounded to 8), `SWEEP_REPEATS`, `SWEEP_EXTRA_EVAL_QUERIES`, `SPOTCHECK_N`.

In [ ]:
MNT_PATHS = find_key_paths(local_cfg, "max_new_tokens")
if not RUN_SWEEP:
    check("sweep_override_effective", "SKIP", "RUN_SWEEP=False"); SWEEP_RAW = pd.DataFrame()
elif not MNT_PATHS:
    other = [p for p in (LOCAL_RUN / "src").rglob("*.py") if "max_new_tokens" in p.read_text(errors="ignore")]
    check("sweep_override_effective", "SKIP", f"'max_new_tokens' not in rag_config; it appears in {[str(p.relative_to(LOCAL_RUN)) for p in other]} — move it to the config to sweep")
    SWEEP_RAW = pd.DataFrame()
else:
    BASE_MNT = int(get_path(local_cfg, MNT_PATHS[0]))
    r8 = lambda x: max(32, int(round(x / 8)) * 8)
    VALUES = sorted({BASE_MNT, *(SWEEP_VALUES or [r8(BASE_MNT * f) for f in (1.28, 0.85, 0.7, 0.55)])}, reverse=True)
    sweep_q = EXAMPLES * SWEEP_REPEATS + [c["query"] for c in CASES if c["source"] == "phase6"][:SWEEP_EXTRA_EVAL_QUERIES]
    spot_q = list(dict.fromkeys(EXAMPLES + [c["query"] for c in CASES if c["source"] == "phase6"]))[:SPOTCHECK_N]
    print("config path:", "/".join(map(str, MNT_PATHS[0])), "| baseline:", BASE_MNT, "| values:", VALUES, "|", len(sweep_q), "streams each")
    rows, SPOT_ANSWERS = [], {}
    try:
        for v in VALUES:
            cfg_v = json.loads(ORIGINAL_LOCAL_CFG_TEXT)
            for pth in MNT_PATHS: set_path(cfg_v, pth, v)
            LOCAL_CFG_PATH.write_text(json.dumps(cfg_v, indent=2))
            SERVER.stop(); SERVER, CLIENT, _ = start_api()
            def sw(q, i, v=v):
                s = CLIENT.recommend_stream(q, f"{SESSION_PREFIX}-sweep{v}-{i}")
                meta = s["meta"] if isinstance(s["meta"], dict) else {}
                done = s["done"] if isinstance(s["done"], dict) else {}
                prods = meta.get("products") or done.get("products") or []
                no_match = bool(meta.get("no_match", done.get("no_match", False)))
                px = Q.answer_proxies(s["answer"], prods, no_match)
                if not prods and not no_match:   # stream did not expose products → citation checks unknown
                    px.update(citation_valid_rate=np.nan, coverage=np.nan, hallucinated_ids=[], cites_any=np.nan)
                return {"mnt": v, "ttft_ms": s["ttft_ms"], "full_ms": s["total_ms"], "answer": s["answer"],
                        "products": prods, "error": s["error"], **px}
            rows += LT.run_requests(sw, sweep_q, len(sweep_q), warmup=2, desc=f"max_new_tokens={v}")
            for q in spot_q:
                SPOT_ANSWERS.setdefault(q, {})[v] = next((r for r in rows if r.get("mnt") == v and r["query"] == q), {})
    finally:
        LOCAL_CFG_PATH.write_text(ORIGINAL_LOCAL_CFG_TEXT)
        print("local config restored to max_new_tokens =", BASE_MNT, "→ restarting the API at the baseline")
        SERVER.stop(); SERVER, CLIENT, _ = start_api()
    SWEEP_RAW = pd.DataFrame(rows)
    SWEEP_RAW.drop(columns=["products"], errors="ignore").to_csv(RESULTS_DIR / "sweep_raw.csv", index=False)

    # did the override take effect? (≈1.33 tokens per English word)
    eff = []
    for v in [x for x in VALUES if x < BASE_MNT]:
        est = SWEEP_RAW.loc[SWEEP_RAW["mnt"] == v, "answer_words"] * 1.33
        eff.append((est > v * 1.25).mean())
    bad = max(eff) if eff else 0
    check("sweep_override_effective", "PASS" if bad < 0.2 else "WARN",
          "answers respect the token cap" if bad < 0.2 else f"{bad:.0%} of answers exceed the cap — the API may not read max_new_tokens from rag_config")

### 📘 9.2 Pick the recommended `max_new_tokens`

**Purpose:** Per value: TTFT and full-answer P50/P95/P99, answer length and the automatic quality checks, each compared with the baseline. Rule: the **largest** value whose full-answer P95 is under the target **and** whose quality deltas stay within `SWEEP_TOLERANCE` (truncation +5 pp, citation validity −0.02, coverage −0.10, cites-any −5 pp).

**Expected Results:** Sweep table with `p95_ok` / `quality_ok`; `sweep_p95_under_target` and `sweep_quality_holds` checks; `RECOMMENDED_MNT`.

**Parameters:** `LATENCY_TARGET_MS`, `SWEEP_HEADROOM_MS`, `SWEEP_TOLERANCE`.

In [ ]:
RECOMMENDED_MNT = None
if len(SWEEP_RAW):
    rows = []
    for v, g in SWEEP_RAW[SWEEP_RAW["ok"]].groupby("mnt"):
        pr = Q.summarize(g)
        rows.append({"max_new_tokens": v, "n": len(g),
                     **{f"ttft_p{p}": LT.percentiles(g["ttft_ms"].astype(float), (p,))[f"p{p}"] for p in (50, 95)},
                     **{f"full_p{p}": LT.percentiles(g["full_ms"].astype(float), (p,))[f"p{p}"] for p in (50, 95, 99)},
                     "mean_words": pr["mean_words"], "truncation_rate": pr["truncation_rate"],
                     "citation_valid_rate": pr["citation_valid_rate"], "coverage": pr["coverage"],
                     "cites_any_rate": pr["cites_any_rate"], "hallucinated_id_rate": pr["hallucinated_id_rate"]})
    SWEEP = pd.DataFrame(rows).sort_values("max_new_tokens", ascending=False).reset_index(drop=True)
    base = SWEEP[SWEEP["max_new_tokens"] == BASE_MNT].iloc[0]
    def quality_ok(r):
        for k, tol in SWEEP_TOLERANCE.items():
            if pd.isna(r[k]) or pd.isna(base[k]): continue
            if tol > 0 and r[k] - base[k] > tol: return False
            if tol < 0 and r[k] - base[k] < tol: return False
        return True
    SWEEP["p95_ok"] = SWEEP["full_p95"] < LATENCY_TARGET_MS - SWEEP_HEADROOM_MS
    SWEEP["quality_ok"] = SWEEP.apply(quality_ok, axis=1)
    display(SWEEP.round(3))
    SWEEP.to_csv(RESULTS_DIR / "sweep_summary.csv", index=False)
    good = SWEEP[SWEEP["p95_ok"] & SWEEP["quality_ok"]]
    RECOMMENDED_MNT = int(good["max_new_tokens"].max()) if len(good) else None
    if len(SWEEP) < 2:
        check("sweep_p95_under_target", "WARN", f"only max_new_tokens={BASE_MNT} was measured — nothing to compare; check SWEEP_VALUES")
        check("sweep_quality_holds", "WARN", "single value, no comparison")
    elif RECOMMENDED_MNT is None:
        check("sweep_p95_under_target", "FAIL" if not SWEEP["p95_ok"].any() else "WARN",
              f"no value meets both P95 < {LATENCY_TARGET_MS} ms and the quality tolerance; fastest quality-safe value: "
              f"{int(SWEEP[SWEEP['quality_ok']].sort_values('full_p95')['max_new_tokens'].iloc[0])}" if SWEEP["p95_ok"].any() else
              f"full-answer P95 ≥ {LATENCY_TARGET_MS} ms at every value (min {SWEEP['full_p95'].min():,.0f} ms)")
        check("sweep_quality_holds", "WARN", "no recommendation")
    else:
        rec = SWEEP[SWEEP["max_new_tokens"] == RECOMMENDED_MNT].iloc[0]
        check("sweep_p95_under_target", "PASS", f"max_new_tokens={RECOMMENDED_MNT}: full-answer P95 {rec['full_p95']:,.0f} ms "
              f"(baseline {BASE_MNT}: {base['full_p95']:,.0f} ms; tried {list(SWEEP['max_new_tokens'])})")
        check("sweep_quality_holds", "PASS", f"auto checks within tolerance (truncation {rec['truncation_rate']:.1%} vs {base['truncation_rate']:.1%}); "
              "confirm with the blind spot-check in 9.3")
    save_json({"baseline": BASE_MNT, "recommended": RECOMMENDED_MNT, "table": SWEEP.to_dict("records")}, "sweep_recommendation.json")

### 📘 9.3 Blind spot-check: baseline vs recommended (interactive)

**Purpose:** Automatic checks miss subtle losses, so rate `SPOTCHECK_N` queries answered at the baseline and at the recommended value, shuffled and unlabeled (the variant key is stored separately).

**Expected Results:** A rating widget over `sweep_spotcheck_sheet.csv` (2 × `SPOTCHECK_N` rows). Skipped when the recommendation equals the baseline.

**Parameters:** `SPOTCHECK_N`.

In [ ]:
SPOT_SHEET, SPOT_KEY = RESULTS_DIR / "sweep_spotcheck_sheet.csv", RESULTS_DIR / "sweep_spotcheck_key.csv"
if RECOMMENDED_MNT is None or RECOMMENDED_MNT == globals().get("BASE_MNT"):
    print("No spot-check needed (no recommendation, or the baseline is already recommended).")
else:
    if not SPOT_SHEET.exists():
        rows = []
        for q, by_v in SPOT_ANSWERS.items():
            for v in (BASE_MNT, RECOMMENDED_MNT):
                r = by_v.get(v) or {}
                rows.append({"_mnt": v, "source": "spotcheck", "query": q,
                             "products_text": products_text(r.get("products") or []), "answer": r.get("answer", ""),
                             "relevance": None, "groundedness": None, "fluency": None, "notes": ""})
        random.Random(7).shuffle(rows)          # ids are assigned after shuffling, so they reveal nothing
        for j, r in enumerate(rows): r["case_id"] = f"spot{j:02d}"
        pd.DataFrame([{"case_id": r["case_id"], "max_new_tokens": r["_mnt"]} for r in rows]).to_csv(SPOT_KEY, index=False)
        pd.DataFrame([{k: v for k, v in r.items() if k != "_mnt"} for r in rows])[
            ["case_id", "source", "query", "products_text", "answer", "relevance", "groundedness", "fluency", "notes"]
        ].to_csv(SPOT_SHEET, index=False)
    _ = R.make_rater(SPOT_SHEET)

### 📘 9.4 Spot-check result and (optionally) apply the recommendation

**Purpose:** Un-blind the spot-check and compare mean ratings; if `APPLY_RECOMMENDED=True`, write the recommended value into the Drive `rag_config.json` (a timestamped backup is written next to it).

**Expected Results:** Mean ratings baseline vs recommended; `sweep_manual_spotcheck` = PASS when no dimension drops by more than 0.3, PENDING until rated.

**Parameters:** `APPLY_RECOMMENDED`.

In [ ]:
if SPOT_KEY.exists() and SPOT_SHEET.exists():
    sp = pd.read_csv(SPOT_SHEET).merge(pd.read_csv(SPOT_KEY), on="case_id")
    sp_r = sp.dropna(subset=DIMS)
    if len(sp_r) >= len(sp) * 0.8:
        means = sp_r.groupby("max_new_tokens")[DIMS].mean()
        display(means.round(2))
        drop = (means.loc[BASE_MNT] - means.loc[RECOMMENDED_MNT]).max()
        check("sweep_manual_spotcheck", "PASS" if drop <= 0.3 else "WARN", f"max mean-rating drop {drop:+.2f} at {RECOMMENDED_MNT} vs {BASE_MNT}")
    else:
        check("sweep_manual_spotcheck", "PENDING", f"{len(sp_r)}/{len(sp)} spot-check rows rated")
elif RUN_SWEEP:
    check("sweep_manual_spotcheck", "SKIP", "baseline already recommended or no recommendation")

if APPLY_RECOMMENDED and RECOMMENDED_MNT and RECOMMENDED_MNT != globals().get("BASE_MNT"):
    drive_cfg_path = REPO_LINK / RAG_CONFIG_REL
    backup = drive_cfg_path.with_name(f"rag_config.json.bak-phase9-{RUN_ID}")
    shutil.copy2(drive_cfg_path, backup)
    cfg = json.loads(drive_cfg_path.read_text())
    for pth in find_key_paths(cfg, "max_new_tokens"): set_path(cfg, pth, RECOMMENDED_MNT)
    drive_cfg_path.write_text(json.dumps(cfg, indent=2))
    print(f"✅ Drive rag_config: max_new_tokens {BASE_MNT} → {RECOMMENDED_MNT} (backup {backup.name})")
else:
    print("Drive config unchanged", "(APPLY_RECOMMENDED=False)." if not APPLY_RECOMMENDED else "(nothing to apply).")

## 10. Failure cases & improvement plan (plan 9.4)

### 📘 10.1 Tag every case

**Purpose:** Join API responses, retrieval scores, answer checks and manual ratings per case and tag failures: retrieval miss, low attribute precision, false / missed no-match, unmet constraints, hallucinated or missing citations, truncation, slow, low manual ratings, user thumbs-down.

**Expected Results:** `results/phase9/failure_cases.csv` and a count per category.

**Parameters:** `PRIMARY_K`, `LATENCY_TARGET_MS`.

In [ ]:
perq = API_PERQ.set_index("qid") if len(API_PERQ) else pd.DataFrame()
prox = PROXY.set_index("case_id")
rat = pd.read_csv(RATING_SHEET).set_index("case_id")[DIMS] if RATING_SHEET.exists() else pd.DataFrame()
case_meta = {c["case_id"]: c for c in CASES}
rows = []
for r in API_RUNS:
    cid = r["case_id"]; c = case_meta[cid]
    row = {"case_id": cid, "source": r["source"], "query": r["query"], "answer": r["answer"],
           "top_titles": " | ".join(str(p.get("title", ""))[:50] for p in r["products"][:3]),
           "no_match": r["no_match"], "unmet_constraints": r["unmet_constraints"], "client_ms": r["client_ms"],
           "expect_no_match": c["expect_no_match"], "reason": r["source"].split(":", 1)[1] if ":" in r["source"] else ""}
    if cid in perq.index: row.update(perq.loc[cid].to_dict())
    if cid in prox.index: row.update({k: v for k, v in prox.loc[cid].to_dict().items() if k != "source"})
    if cid in rat.index: row.update(rat.loc[cid].to_dict())
    row["tags"] = F.classify(row, PRIMARY_K, LATENCY_TARGET_MS)
    rows.append(row)
FAIL_DF = pd.DataFrame(rows)
FAIL_DF.assign(tags=FAIL_DF["tags"].map(";".join)).to_csv(RESULTS_DIR / "failure_cases.csv", index=False)
counts = FAIL_DF.explode("tags")["tags"].value_counts()
display(counts.rename("cases").to_frame())
print(f"{FAIL_DF['tags'].map(bool).sum()}/{len(FAIL_DF)} cases have ≥ 1 failure tag")

### 📘 10.2 Write `failure_cases.md`

**Purpose:** Render the write-up the plan explicitly asks for: headline numbers, retrieval / latency / sweep tables, the failure taxonomy, examples per category and a ranked improvement plan (cases × severity).

**Expected Results:** `results/phase9/failure_cases.md`, displayed below. **Edit the improvement plan by hand** — the fixes are generated starting points. `failure_writeup` = PASS.

**Parameters:** none

In [ ]:
headline = {}
if RUN_RETRIEVAL_COMPARISON:
    rc = RET_CMP.set_index("metric")
    headline[f"in-process {primary} pretrained → finetuned"] = f"{rc.loc[primary, 'pretrained']:.3f} → {rc.loc[primary, 'finetuned']:.3f}"
headline[f"API {primary_api}"] = round(API_MEAN.get(primary_api, np.nan), 3)
headline["no-match accuracy"] = round(NO_MATCH_ACC, 3) if not np.isnan(NO_MATCH_ACC) else "n/a"
if len(RATING_SUM): headline.update({f"manual {d} (mean, n={len(rated)})": round(RATING_SUM.loc[d, "mean"], 2) for d in DIMS})
headline["hallucinated-id rate"] = f"{PROXY_SUM['hallucinated_id_rate']:.1%}"
headline["client P95 (ms)"] = round(p95)
headline["stream full-answer P95 (ms)"] = round(fp95)
headline["recommended max_new_tokens"] = RECOMMENDED_MNT if RECOMMENDED_MNT else "none"
sections = {"Retrieval: pretrained vs fine-tuned (in-process)": RET_CMP.round(4) if RUN_RETRIEVAL_COMPARISON else pd.DataFrame(),
            "Retrieval through the API vs Phase 6": API_CMP.round(4),
            "Latency (sequential, ms)": LAT_TABLE, "Streaming (Phase 8 examples, ms)": STREAM_TABLE,
            "max_new_tokens sweep": SWEEP.round(3) if len(SWEEP_RAW) else pd.DataFrame(),
            "Production logs (test sessions removed)": pd.DataFrame([LOG_STATS]).T.reset_index().rename(columns={"index": "stat", 0: "value"})}
REPORT = F.build_report(FAIL_DF, headline, sections)
(RESULTS_DIR / "failure_cases.md").write_text(REPORT)
display(Markdown(REPORT))
check("failure_writeup", "PASS", f"failure_cases.md with {len(counts)} categories")

## 11. Exit checklist

### 📘 11.1 Stop the API, archive Phase 9 logs, save the checklist

**Purpose:** Stop uvicorn, copy the Phase 9 request/feedback logs and uvicorn log to `results/phase9/api_logs/`, save `summary.json` (the latency baseline Phase 10 adds speech-to-text to) and `exit_checklist.json/.md`.

**Expected Results:** A table of every check. Phase 9 is **done** when there is no FAIL and no PENDING; WARNs need a sentence in the write-up.

**Parameters:** none

In [ ]:
try: SERVER.stop()
except Exception as e: print("stop:", e)
arch = RESULTS_DIR / "api_logs"; arch.mkdir(exist_ok=True)
for f in (LOCAL_RUN / "logs").glob("*"):
    if f.is_file(): shutil.copy2(f, arch / f.name)

summary = {"run_id": RUN_ID, "eval_file": str(EVAL_PATH), "n_cases": len(CASES), "fake": USE_FAKE,
           "retrieval": RET_CMP.to_dict("records") if RUN_RETRIEVAL_COMPARISON else None,
           "api_retrieval": API_MEAN, "no_match_accuracy": NO_MATCH_ACC, "answer_checks": PROXY_SUM,
           "manual": RATING_SUM["mean"].to_dict() if len(RATING_SUM) else None,
           "gpu": globals().get("GPU_NAME"), "latency_ms": LAT_TABLE.to_dict("records"), "stream_ms": STREAM_TABLE.to_dict("records"),
           "max_new_tokens": {"baseline": globals().get("BASE_MNT"), "recommended": RECOMMENDED_MNT},
           "log_stats": LOG_STATS}
save_json(summary, "summary.json")

order = ["eval_set_loaded", "phase6_baseline_found", "log_cases_mined", "chroma_count", "index_reembed_cosine",
         "retrieval_ft_beats_pretrained", "phase6_reproduced", "chroma_consistency", "api_health_ok", "api_schema",
         "api_eval_errors", "api_retrieval_vs_phase6", "no_hallucinated_ids", "citations_present",
         "manual_ratings_count", "manual_quality", "latency_n", "latency_p95_client", "latency_stage_breakdown", "latency_vs_phase6",
         "stream_full_p95_baseline", "sweep_override_effective", "sweep_p95_under_target", "sweep_quality_holds",
         "sweep_manual_spotcheck", "failure_writeup"]
EXIT = pd.DataFrame([{"check": k, **CHECKS.get(k, {"status": "MISSING", "detail": "cell not run"})} for k in order] +
                    [{"check": k, **v} for k, v in CHECKS.items() if k not in order])
EXIT["status"] = EXIT["status"].map(lambda s: f"{_ICON.get(s, '❓')} {s}")
display(EXIT)
save_json({"run_id": RUN_ID, "checks": CHECKS}, "exit_checklist.json")
(RESULTS_DIR / "exit_checklist.md").write_text(
    "# Phase 9 exit checklist\n\n| check | status | detail |\n|---|---|---|\n" +
    "\n".join(f"| {r.check} | {r.status} | {str(r.detail).replace('|', '/')} |" for r in EXIT.itertuples()))
statuses = [v["status"] for v in CHECKS.values()]
blocking = statuses.count("FAIL") + statuses.count("PENDING") + sum(k not in CHECKS for k in order)
print(f"\nPASS {statuses.count('PASS')} · WARN {statuses.count('WARN')} · FAIL {statuses.count('FAIL')} · "
      f"PENDING {statuses.count('PENDING')} · SKIP {statuses.count('SKIP')}")
print("✅ Phase 9 complete" if blocking == 0 else "⏳ Not done yet: resolve FAIL / PENDING / MISSING items above")
print("Results:", RESULTS_DIR)